# E-Commerce Data Warehouse Pipeline

This notebook implements a complete data warehouse pipeline for the Udacity Data Engineering course.

**Business Context:** You are a data engineer at a fast-growing e-commerce company. Critical data is spread across multiple operational systems (PostgreSQL, Cassandra, Neo4j), making it difficult for analysts to run consistent reports. Your job is to design and implement a centralized analytics warehouse in Amazon Redshift.

## Tasks Overview

1. **Explore and Plan** - Review CSV data, identify key fields, map to warehouse schema
2. **Design the Schema** - Create dimensional model with staging, dimension, and fact tables
3. **Extract and Transform** - Load source systems and extract/transform data
4. **Load into Redshift** - Execute DDL, load staging tables, populate dimensions and facts
5. **Optimize Performance** - Apply best practices, create materialized views
6. **Validate and Report** - Run quality checks, generate final report

---
## The Data

The dataset consists of three CSV files representing data from different operational systems:

### Orders Data (PostgreSQL source) - `ecom_orders_postgres.csv`
- **order_id**: Unique identifier for each order
- **customer_id**: Customer who placed the order
- **order_datetime / ship_datetime**: Timestamps for order and shipping
- **channel / device_type / browser**: How the order was placed
- **country / state**: Geographic location
- **payment_method / campaign**: Payment and marketing info
- **Financial fields**: subtotal, discount, shipping, tax, total amounts
- **Delivery fields**: delivery_days, on_time_delivery
- **Flags**: authorization_approved, returned

### Events Data (Cassandra source) - `ecom_events_cassandra.csv`
- **event_id / session_id**: Event and session identifiers
- **customer_id**: Customer who triggered the event
- **event_type**: Type of event (page_view, product_view, add_to_cart, etc.)
- **event_ts**: Timestamp of the event
- **Device/browser/OS info**: Technical context
- **Behavioral fields**: page_depth, latency_ms, dwell_seconds
- **Commerce fields**: cart_value_usd, discount_rate, fraud_score

### Graph Edges Data (Neo4j source) - `ecom_graph_edges_neo4j.csv`
- **edge_id**: Unique relationship identifier
- **from_node_id / to_node_id**: Source and target nodes
- **from_node_type / to_node_type**: Node types (Customer, Product, Order)
- **relationship**: Type of relationship (PURCHASED, VIEWED, ADDED_TO_CART, etc.)
- **Context fields**: order_id, category, customer_segment, campaign
- **Metrics**: edge_strength, unit_price_usd, quantity

---
## Setup: Imports and Dependencies

Run this cell first to import all required libraries.

In [21]:
# ========= Imports
import os, io, re, time, json, textwrap
from datetime import datetime
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd

# Source system libraries
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from neo4j import GraphDatabase

# Warehouse (Redshift) via Data API
import boto3

# Optional progress bars
try:
    from tqdm import tqdm
    TQDM = True
except Exception:
    TQDM = False

print("All imports successful!")
print(f"   - pandas version: {pd.__version__}")
print(f"   - numpy version: {np.__version__}")

All imports successful!
   - pandas version: 2.3.1
   - numpy version: 2.2.6


---
## Setup: Configuration

Update these settings for your environment. You will need to:
1. Set your AWS credentials (from Cloud Resources)
2. Configure database connection parameters

In [ ]:
# Set up AWS credentials for the session (get these from Cloud Resources)
# IMPORTANT: Replace with your actual credentials
os.environ['AWS_ACCESS_KEY_ID'] = 'A***************'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'J***********************'
os.environ['AWS_SESSION_TOKEN'] = 'I************************'

In [23]:
# ========= Configuration
BASE_DIR = os.getenv("PROJECT_BASE_DIR", ".")
DATA_DIR = os.path.join(BASE_DIR, "data")
CSV_ORDERS  = os.path.join(DATA_DIR, "ecom_orders_postgres.csv")
CSV_EVENTS  = os.path.join(DATA_DIR, "ecom_events_cassandra.csv")
CSV_EDGES   = os.path.join(DATA_DIR, "ecom_graph_edges_neo4j.csv")
DDL_MD_PATH = os.path.join(BASE_DIR, "project-ddl-long.md")
MERMAID_MD  = os.path.join(BASE_DIR, "project-mermaid-diagram.md")
BATCH_SIZE  = int(os.getenv("BATCH_SIZE", "100"))

# PostgreSQL
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB",   "postgres")
PG_USER = os.getenv("PG_USER", "temp")
PG_PW   = os.getenv("PG_PW",   "temp")

# Cassandra
CAS_HOSTS = os.getenv("CAS_HOSTS", "localhost").split(",")
CAS_PORT  = int(os.getenv("CAS_PORT", "9042"))
CAS_USER  = os.getenv("CAS_USER", "")
CAS_PW    = os.getenv("CAS_PW", "")
CAS_KEYSPACE = os.getenv("CAS_KEYSPACE", "ecommerce")

# Neo4j
NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PW   = os.getenv("NEO4J_PW",   "neo4jpass")

# AWS/Redshift
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
REDSHIFT_DATABASE = os.getenv("REDSHIFT_DATABASE", "ecom")
REDSHIFT_WORKGROUP = os.getenv("REDSHIFT_WORKGROUP", "udacity-dwh-wg")
REDSHIFT_SECRET_ARN = os.getenv("REDSHIFT_SECRET_ARN")
REDSHIFT_CLUSTER_IDENTIFIER = os.getenv("REDSHIFT_CLUSTER_IDENTIFIER")
REDSHIFT_DB_USER = os.getenv("REDSHIFT_DB_USER")

# Verify configuration
print("Configuration loaded!")
print(f"   - BASE_DIR: {BASE_DIR}")
print(f"   - PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"   - Cassandra: {CAS_HOSTS}:{CAS_PORT}/{CAS_KEYSPACE}")
print(f"   - Neo4j: {NEO4J_URI}")
print(f"   - Redshift: {REDSHIFT_DATABASE} (workgroup: {REDSHIFT_WORKGROUP})")

print()
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    print(f"   AWS credentials found (Key ID: {AWS_ACCESS_KEY_ID[:10]}...)")
else:
    print("   WARNING: AWS credentials NOT FOUND - set them above!")

Configuration loaded!
   - BASE_DIR: .
   - PostgreSQL: localhost:5432/postgres
   - Cassandra: ['localhost']:9042/ecommerce
   - Neo4j: bolt://localhost:7687
   - Redshift: ecom (workgroup: udacity-dwh-wg)

   AWS credentials found (Key ID: ASIARKJXVC...)


---
## Setup: Column Specifications

These define the mapping from source columns to Redshift staging tables.
Use these as a reference when building your transformation logic.

In [24]:
# Column specs for Redshift staging (name, kind)
# kind: 's' = string, 'ts' = timestamp, 'i' = integer, 'f' = float, 'b' = boolean

ORDERS_COLSPEC = [
    ('order_id','s'),('customer_id','s'),('order_datetime','ts'),('ship_datetime','ts'),
    ('channel','s'),('device_type','s'),('browser','s'),('country','s'),('state','s'),
    ('payment_method','s'),('campaign','s'),('primary_category','s'),('num_distinct_items','i'),
    ('subtotal_usd','f'),('discount_rate','f'),('discount_amount_usd','f'),('shipping_method','s'),
    ('shipping_cost_usd','f'),('tax_rate','f'),('tax_amount_usd','f'),('order_total_usd','f'),
    ('order_weight_kg','f'),('delivery_days','i'),('on_time_delivery','b'),
    ('authorization_approved','b'),('returned','b')
]

EVENTS_COLSPEC = [
    ('event_id','s'),('customer_id','s'),('session_id','s'),('event_type','s'),('event_ts','ts'),
    ('device_type','s'),('browser','s'),('os','s'),('referrer','s'),('country','s'),('state','s'),
    ('ab_variant','s'),('is_logged_in','b'),('page_depth','i'),('latency_ms','i'),
    ('dwell_seconds','i'),('cart_value_usd','f'),('discount_rate','f'),('fraud_score','f'),
    ('payment_outcome','s'),('sequence_num','i'),('product_id','s'),('category','s'),('promo_code','s')
]

EDGES_COLSPEC = [
    ('edge_id','s'),('from_node_id','s'),('from_node_type','s'),('to_node_id','s'),('to_node_type','s'),
    ('relationship','s'),('timestamp','ts'),('order_id','s'),('category','s'),('customer_segment','s'),
    ('edge_strength','f'),('price_bucket','s'),('region','s'),('state','s'),('campaign','s'),
    ('same_household','b'),('prior_interactions','i'),('dwell_seconds','i'),('product_id','s'),
    ('unit_price_usd','f'),('quantity','i'),('returned_flag','b'),('auth_approved','b')
]

print(f"Column specs defined:")
print(f"   - Orders: {len(ORDERS_COLSPEC)} columns")
print(f"   - Events: {len(EVENTS_COLSPEC)} columns")
print(f"   - Edges: {len(EDGES_COLSPEC)} columns")

Column specs defined:
   - Orders: 26 columns
   - Events: 24 columns
   - Edges: 23 columns


---
## Setup: Helper Functions

Utility functions used throughout the pipeline.

In [25]:
def trim_df(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize text fields and handle NaN values."""
    df = df.copy()
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({'nan': np.nan, 'None': np.nan, 'NaN': np.nan, '': np.nan})
    return df

def read_csvs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Read all three source CSVs and apply cleaning."""
    orders = pd.read_csv(CSV_ORDERS)
    events = pd.read_csv(CSV_EVENTS)
    edges  = pd.read_csv(CSV_EDGES)
    return trim_df(orders), trim_df(events), trim_df(edges)

print("Helper functions defined: trim_df(), read_csvs()")

Helper functions defined: trim_df(), read_csvs()


---
# Task 1: Explore and Plan the Data Pipeline

In this task, you will:
- Review the provided CSV data files to understand structure, columns, and content
- Identify key fields and relationships important for analysis
- Map source fields to fact and dimension tables
- Consider data format standardization needs

**Deliverables:**
- Written plan mapping fields from all three sources to fact and dimension tables
- Documentation of key relationships and ID standardization strategies

In [26]:
# Read the CSV files
print("Reading CSV files...")
orders_df, events_df, edges_df = read_csvs()

print("\n" + "="*80)
print("TASK 1: DATA EXPLORATION AND PIPELINE PLANNING")
print("="*80)

# ---------------------------------------------------------
# 1. Basic structure
# ---------------------------------------------------------

datasets = {
    "ORDERS DATA - PostgreSQL source": orders_df,
    "EVENTS DATA - Cassandra source": events_df,
    "GRAPH EDGES DATA - Neo4j source": edges_df
}

for name, df in datasets.items():
    print("\n" + "-"*80)
    print(f"📊 {name}")
    print("-"*80)
    print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]:,} columns")
    print("\nColumns:")
    print(list(df.columns))
    print("\nSample rows:")
    display(df.head(5))
    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))

# ---------------------------------------------------------
# 2. Primary keys and uniqueness
# ---------------------------------------------------------

print("\n" + "="*80)
print("PRIMARY KEYS AND UNIQUENESS CHECKS")
print("="*80)

primary_key_checks = [
    ("orders_df", orders_df, "order_id"),
    ("events_df", events_df, "event_id"),
    ("edges_df", edges_df, "edge_id")
]

pk_results = []

for dataset_name, df, pk in primary_key_checks:
    total_rows = len(df)
    unique_values = df[pk].nunique(dropna=True)
    null_values = df[pk].isna().sum()
    duplicate_count = total_rows - unique_values

    pk_results.append({
        "dataset": dataset_name,
        "primary_key": pk,
        "total_rows": total_rows,
        "unique_values": unique_values,
        "null_values": null_values,
        "duplicate_count": duplicate_count,
        "is_unique": duplicate_count == 0 and null_values == 0
    })

pk_results_df = pd.DataFrame(pk_results)
display(pk_results_df)

# ---------------------------------------------------------
# 3. Foreign key / relationship analysis
# ---------------------------------------------------------

print("\n" + "="*80)
print("KEY RELATIONSHIPS ACROSS SOURCES")
print("="*80)

relationship_results = []

# Customer relationship between orders and events
orders_customers = set(orders_df["customer_id"].dropna().astype(str))
events_customers = set(events_df["customer_id"].dropna().astype(str))

relationship_results.append({
    "relationship": "orders.customer_id -> events.customer_id",
    "left_count": len(orders_customers),
    "right_count": len(events_customers),
    "overlap_count": len(orders_customers.intersection(events_customers)),
    "description": "Customer identifier appears in both PostgreSQL orders and Cassandra events."
})

# Order relationship between orders and graph edges
if "order_id" in edges_df.columns:
    edge_orders = set(edges_df["order_id"].dropna().astype(str))
    relationship_results.append({
        "relationship": "edges.order_id -> orders.order_id",
        "left_count": len(edge_orders),
        "right_count": orders_df["order_id"].nunique(dropna=True),
        "overlap_count": len(edge_orders.intersection(set(orders_df["order_id"].dropna().astype(str)))),
        "description": "Graph edges can reference orders from PostgreSQL."
    })

# Product relationship between events and graph edges
if "product_id" in events_df.columns and "product_id" in edges_df.columns:
    events_products = set(events_df["product_id"].dropna().astype(str))
    edge_products = set(edges_df["product_id"].dropna().astype(str))

    relationship_results.append({
        "relationship": "events.product_id -> edges.product_id",
        "left_count": len(events_products),
        "right_count": len(edge_products),
        "overlap_count": len(events_products.intersection(edge_products)),
        "description": "Product identifier appears in behavioral events and graph relationships."
    })

relationship_results_df = pd.DataFrame(relationship_results)
display(relationship_results_df)

# ---------------------------------------------------------
# 4. Date/time fields
# ---------------------------------------------------------

print("\n" + "="*80)
print("DATE/TIME FIELDS FOR DATE DIMENSION")
print("="*80)

date_fields = [
    ("orders_df", "order_datetime"),
    ("orders_df", "ship_datetime"),
    ("events_df", "event_ts"),
    ("edges_df", "timestamp")
]

date_summary = []

for dataset_name, col in date_fields:
    df = {
        "orders_df": orders_df,
        "events_df": events_df,
        "edges_df": edges_df
    }[dataset_name]

    if col in df.columns:
        converted = pd.to_datetime(df[col], errors="coerce")
        date_summary.append({
            "dataset": dataset_name,
            "timestamp_column": col,
            "null_or_invalid_values": converted.isna().sum(),
            "min_timestamp": converted.min(),
            "max_timestamp": converted.max(),
            "date_key_needed": True
        })

date_summary_df = pd.DataFrame(date_summary)
display(date_summary_df)

# ---------------------------------------------------------
# 5. Categorical fields and cardinality
# ---------------------------------------------------------

print("\n" + "="*80)
print("CATEGORICAL FIELDS AND DIMENSION CANDIDATES")
print("="*80)

categorical_candidates = {
    "orders_df": [
        "channel", "device_type", "browser", "country", "state",
        "payment_method", "campaign", "primary_category", "shipping_method"
    ],
    "events_df": [
        "event_type", "device_type", "browser", "os", "referrer",
        "country", "state", "ab_variant", "payment_outcome", "category", "promo_code"
    ],
    "edges_df": [
        "from_node_type", "to_node_type", "relationship", "category",
        "customer_segment", "price_bucket", "region", "state", "campaign"
    ]
}

cardinality_results = []

for dataset_name, columns in categorical_candidates.items():
    df = {
        "orders_df": orders_df,
        "events_df": events_df,
        "edges_df": edges_df
    }[dataset_name]

    for col in columns:
        if col in df.columns:
            cardinality_results.append({
                "dataset": dataset_name,
                "column": col,
                "distinct_values": df[col].nunique(dropna=True),
                "null_values": df[col].isna().sum(),
                "sample_values": ", ".join(df[col].dropna().astype(str).unique()[:5])
            })

cardinality_df = pd.DataFrame(cardinality_results)
display(cardinality_df.sort_values(["dataset", "distinct_values"]))

# ---------------------------------------------------------
# 6. Data quality summary
# ---------------------------------------------------------

print("\n" + "="*80)
print("DATA QUALITY SUMMARY")
print("="*80)

critical_columns = {
    "orders_df": ["order_id", "customer_id", "order_datetime", "order_total_usd"],
    "events_df": ["event_id", "customer_id", "session_id", "event_ts"],
    "edges_df": ["edge_id", "from_node_id", "to_node_id", "relationship", "timestamp"]
}

dq_results = []

for dataset_name, columns in critical_columns.items():
    df = {
        "orders_df": orders_df,
        "events_df": events_df,
        "edges_df": edges_df
    }[dataset_name]

    for col in columns:
        if col in df.columns:
            dq_results.append({
                "dataset": dataset_name,
                "column": col,
                "total_rows": len(df),
                "null_values": df[col].isna().sum(),
                "null_percentage": round(df[col].isna().mean() * 100, 2),
                "distinct_values": df[col].nunique(dropna=True)
            })

dq_results_df = pd.DataFrame(dq_results)
display(dq_results_df)

# ---------------------------------------------------------
# 7. Source-to-target mapping table
# ---------------------------------------------------------

print("\n" + "="*80)
print("SOURCE-TO-TARGET FIELD MAPPING")
print("="*80)

mapping_rows = [
    # Orders source
    {
        "source_system": "PostgreSQL",
        "source_file": "ecom_orders_postgres.csv",
        "source_fields": "order_id, customer_id, order_datetime, ship_datetime, channel, device_type, browser, payment_method, campaign, financial fields",
        "staging_table": "stg.orders_raw",
        "target_tables": "dw.fact_orders, dw.dim_customer, dw.dim_date, dw.dim_channel, dw.dim_device, dw.dim_browser, dw.dim_campaign, dw.dim_payment_method, dw.dim_shipping_method",
        "business_use": "Revenue analysis, delivery performance, payment analysis, campaign performance"
    },
    {
        "source_system": "Cassandra",
        "source_file": "ecom_events_cassandra.csv",
        "source_fields": "event_id, customer_id, session_id, event_type, event_ts, product_id, device/browser/os, referrer, cart and behavioral metrics",
        "staging_table": "stg.events_raw",
        "target_tables": "dw.fact_events, dw.dim_customer, dw.dim_product, dw.dim_date, dw.dim_device, dw.dim_browser, dw.dim_os, dw.dim_referrer, dw.dim_ab_variant",
        "business_use": "Customer behavior analysis, funnel analysis, product engagement, technical performance"
    },
    {
        "source_system": "Neo4j",
        "source_file": "ecom_graph_edges_neo4j.csv",
        "source_fields": "edge_id, from_node_id, to_node_id, relationship, timestamp, order_id, product_id, edge_strength, customer_segment",
        "staging_table": "stg.edges_raw",
        "target_tables": "dw.fact_graph_edges, dw.dim_customer, dw.dim_product, dw.dim_date, dw.dim_campaign",
        "business_use": "Recommendation graph analysis, product relationship analysis, customer-product interaction analysis"
    }
]

mapping_df = pd.DataFrame(mapping_rows)
display(mapping_df)

# ---------------------------------------------------------
# 8. Written planning summary for the notebook
# ---------------------------------------------------------

pipeline_plan = """
## Task 1 Pipeline Planning Summary

### Source Systems
The warehouse integrates three heterogeneous operational sources:

1. PostgreSQL orders data:
   - Transactional order and payment data.
   - Main identifier: order_id.
   - Important foreign key: customer_id.
   - Main fact target: dw.fact_orders.

2. Cassandra events data:
   - Real-time clickstream and behavioral activity.
   - Main identifier: event_id.
   - Important identifiers: customer_id, session_id, product_id.
   - Main fact target: dw.fact_events.

3. Neo4j graph edges data:
   - Product, customer, and recommendation graph relationships.
   - Main identifier: edge_id.
   - Important identifiers: from_node_id, to_node_id, order_id, product_id.
   - Main fact target: dw.fact_graph_edges.

### Identifier Conformance Strategy
- customer_id is standardized as the shared customer business key across orders and events.
- product_id is standardized as the shared product business key across events and graph edges.
- order_id is used to connect graph relationships back to transactional orders when available.
- Surrogate keys are generated in Redshift dimensions, such as customer_sk and product_sk, to support a clean dimensional model.

### Date Standardization Strategy
- order_datetime, ship_datetime, event_ts, and timestamp are converted into date keys using YYYYMMDD format.
- These date keys join to dw.dim_date for consistent time-based reporting.

### Dimensional Modeling Strategy
- Staging tables preserve raw source structure.
- Dimension tables standardize customers, products, channels, devices, browsers, campaigns, dates, OS, referrers, shipping methods, payment methods, and A/B variants.
- Fact tables store measurable business events at clear grains:
  - fact_orders: one row per order.
  - fact_events: one row per clickstream event.
  - fact_graph_edges: one row per graph relationship.

### Analytics Supported
The model supports:
- Daily revenue and average order value.
- Customer behavior and funnel analysis.
- Product performance and recommendation analysis.
- Campaign and channel performance.
- Delivery and return analysis.
"""

print(pipeline_plan)

Reading CSV files...

TASK 1: DATA EXPLORATION AND PIPELINE PLANNING

--------------------------------------------------------------------------------
📊 ORDERS DATA - PostgreSQL source
--------------------------------------------------------------------------------
Shape: 2,500 rows, 26 columns

Columns:
['order_id', 'customer_id', 'order_datetime', 'ship_datetime', 'channel', 'device_type', 'browser', 'country', 'state', 'payment_method', 'campaign', 'primary_category', 'num_distinct_items', 'subtotal_usd', 'discount_rate', 'discount_amount_usd', 'shipping_method', 'shipping_cost_usd', 'tax_rate', 'tax_amount_usd', 'order_total_usd', 'order_weight_kg', 'delivery_days', 'on_time_delivery', 'authorization_approved', 'returned']

Sample rows:


,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
0,ORD100000,C29457,2024-06-12 02:14:21,2024-06-17 02:14:21,android_app,mobile,Safari,CA,CT,google_pay,...,standard,5.05,0.0000,0.00,838.39,0.10,5,True,True,False
1,ORD100001,C22666,2024-08-08 00:16:41,2024-08-11 00:16:41,mobile_web,desktop,Opera,DE,OH,apple_pay,...,standard,9.35,0.0000,0.00,388.60,0.20,3,True,True,False
2,ORD100002,C72623,2024-09-11 06:59:18,2024-09-15 06:59:18,web,desktop,Edge,US,TX,google_pay,...,standard,6.63,0.0923,59.32,708.60,0.85,4,True,True,False
3,ORD100003,C62733,2025-03-13 15:54:14,2025-03-17 15:54:14,web,mobile,Safari,UK,VT,google_pay,...,standard,7.15,0.0000,0.00,104.23,0.10,4,True,True,False
4,ORD100004,C62083,2025-02-28 00:20:21,2025-03-06 00:20:21,android_app,tablet,Opera,US,TN,apple_pay,...,standard,3.96,0.0864,3.69,50.41,0.29,6,False,True,False



Data types:


,dtype
order_id,object
customer_id,object
order_datetime,object
ship_datetime,object
channel,object
device_type,object
browser,object
country,object
state,object
payment_method,object



--------------------------------------------------------------------------------
📊 EVENTS DATA - Cassandra source
--------------------------------------------------------------------------------
Shape: 2,500 rows, 24 columns

Columns:
['event_id', 'customer_id', 'session_id', 'event_type', 'event_ts', 'device_type', 'browser', 'os', 'referrer', 'country', 'state', 'ab_variant', 'is_logged_in', 'page_depth', 'latency_ms', 'dwell_seconds', 'cart_value_usd', 'discount_rate', 'fraud_score', 'payment_outcome', 'sequence_num', 'product_id', 'category', 'promo_code']

Sample rows:


...


DATA QUALITY SUMMARY


,dataset,column,total_rows,null_values,null_percentage,distinct_values
0,orders_df,order_id,2500,0,0.0,2500
1,orders_df,customer_id,2500,0,0.0,2460
2,orders_df,order_datetime,2500,0,0.0,2500
3,orders_df,order_total_usd,2500,0,0.0,2445
4,events_df,event_id,2500,0,0.0,2500
5,events_df,customer_id,2500,0,0.0,2461
6,events_df,session_id,2500,0,0.0,2500
7,events_df,event_ts,2500,0,0.0,2500
8,edges_df,edge_id,2500,0,0.0,2500
9,edges_df,from_node_id,2500,0,0.0,2474



SOURCE-TO-TARGET FIELD MAPPING


,source_system,source_file,source_fields,staging_table,target_tables,business_use
0,PostgreSQL,ecom_orders_postgres.csv,"order_id, customer_id, order_datetime, ship_da...",stg.orders_raw,"dw.fact_orders, dw.dim_customer, dw.dim_date, ...","Revenue analysis, delivery performance, paymen..."
1,Cassandra,ecom_events_cassandra.csv,"event_id, customer_id, session_id, event_type,...",stg.events_raw,"dw.fact_events, dw.dim_customer, dw.dim_produc...","Customer behavior analysis, funnel analysis, p..."
2,Neo4j,ecom_graph_edges_neo4j.csv,"edge_id, from_node_id, to_node_id, relationshi...",stg.edges_raw,"dw.fact_graph_edges, dw.dim_customer, dw.dim_p...","Recommendation graph analysis, product relatio..."



## Task 1 Pipeline Planning Summary

### Source Systems
The warehouse integrates three heterogeneous operational sources:

1. PostgreSQL orders data:
   - Transactional order and payment data.
   - Main identifier: order_id.
   - Important foreign key: customer_id.
   - Main fact target: dw.fact_orders.

2. Cassandra events data:
   - Real-time clickstream and behavioral activity.
   - Main identifier: event_id.
   - Important identifiers: customer_id, session_id, product_id.
   - Main fact target: dw.fact_events.

3. Neo4j graph edges data:
   - Product, customer, and recommendation graph relationships.
   - Main identifier: edge_id.
   - Important identifiers: from_node_id, to_node_id, order_id, product_id.
   - Main fact target: dw.fact_graph_edges.

### Identifier Conformance Strategy
- customer_id is standardized as the shared customer business key across orders and events.
- product_id is standardized as the shared product business key across events and graph edges.
- order_id i

In [27]:
# Explore the events data
print(f"\n📊 EVENTS DATA (from Cassandra)")
print(f"   Shape: {events_df.shape[0]} rows, {events_df.shape[1]} columns")

# Print columns
print("\nColumns:")
print(events_df.columns.tolist())

# Display sample rows
print("\nSample rows:")
display(events_df.head(5))

# Optional (muy recomendado para nota alta): tipos de datos
print("\nData types:")
display(events_df.dtypes.to_frame("dtype"))


📊 EVENTS DATA (from Cassandra)
   Shape: 2500 rows, 24 columns

Columns:
['event_id', 'customer_id', 'session_id', 'event_type', 'event_ts', 'device_type', 'browser', 'os', 'referrer', 'country', 'state', 'ab_variant', 'is_logged_in', 'page_depth', 'latency_ms', 'dwell_seconds', 'cart_value_usd', 'discount_rate', 'fraud_score', 'payment_outcome', 'sequence_num', 'product_id', 'category', 'promo_code']

Sample rows:


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200000,C47857,S7645111197,page_view,2024-11-14 00:17:09,desktop,Firefox,Linux,organic_search,IN,...,331,35,18.12,0.165,0.000,NaN,3,NaN,NaN,NONE
1,EVT200001,C83195,S1423573902,page_view,2024-08-15 10:42:18,mobile,Chrome,Linux,social,DE,...,432,26,33.62,0.083,0.262,NaN,2,NaN,NaN,NONE
2,EVT200002,C27664,S2829853742,product_view,2024-05-03 23:28:24,mobile,Chrome,Android,email,BR,...,360,6,12.26,0.237,0.162,NaN,5,P5735,Home,FREESHIP
3,EVT200003,C55911,S2500554740,add_to_cart,2024-10-28 18:44:50,desktop,Firefox,Linux,direct,US,...,414,22,86.55,0.197,0.067,NaN,1,P1730,Pets,FREESHIP
4,EVT200004,C27347,S3072616664,add_to_cart,2024-07-22 07:43:54,mobile,Chrome,Linux,direct,CA,...,381,32,87.78,0.058,0.000,NaN,8,P6252,Pets,WELCOME10



Data types:


,dtype
event_id,object
customer_id,object
session_id,object
event_type,object
event_ts,object
device_type,object
browser,object
os,object
referrer,object
country,object


In [28]:
# Explore the graph edges data
print(f"\n📊 GRAPH EDGES DATA (from Neo4j)")
print(f"   Shape: {edges_df.shape[0]} rows, {edges_df.shape[1]} columns")

# Print columns
print("\nColumns:")
print(edges_df.columns.tolist())

# Display sample rows
print("\nSample rows:")
display(edges_df.head(5))

# Optional (muy recomendado): tipos de datos
print("\nData types:")
display(edges_df.dtypes.to_frame("dtype"))


📊 GRAPH EDGES DATA (from Neo4j)
   Shape: 2500 rows, 23 columns

Columns:
['edge_id', 'from_node_id', 'from_node_type', 'to_node_id', 'to_node_type', 'relationship', 'timestamp', 'order_id', 'category', 'customer_segment', 'edge_strength', 'price_bucket', 'region', 'state', 'campaign', 'same_household', 'prior_interactions', 'dwell_seconds', 'product_id', 'unit_price_usd', 'quantity', 'returned_flag', 'auth_approved']

Sample rows:


,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE300000,C65482,Customer,P2261,Product,RETURNS,2024-01-27 13:42:26,ORD100938,Books,New,...,WA,SpringSale,False,3,66,P2261,315.87,2,False,False
1,EDGE300001,C59599,Customer,P7625,Product,PURCHASED,2025-02-07 10:22:55,ORD102431,Toys,Loyal,...,WI,Holiday,False,4,62,P7625,19.21,1,False,True
2,EDGE300002,C44719,Customer,P8677,Product,ADDED_TO_CART,2024-01-06 14:04:37,NaN,Pets,Active,...,KS,Loyalty,False,5,16,P8677,68.68,2,False,False
3,EDGE300003,C13897,Customer,P7883,Product,VIEWED,2024-06-04 21:36:14,NaN,Grocery,Active,...,NV,Clearance,False,3,44,P7883,74.70,1,False,False
4,EDGE300004,C30350,Customer,P2015,Product,ADDED_TO_CART,2024-06-23 20:00:17,NaN,Toys,Active,...,AR,SpringSale,False,2,17,P2015,52.71,1,False,False



Data types:


,dtype
edge_id,object
from_node_id,object
from_node_type,object
to_node_id,object
to_node_type,object
relationship,object
timestamp,object
order_id,object
category,object
customer_segment,object


In [29]:
# Identify key fields and relationships
print("\n" + "="*60)
print("KEY FIELDS AND RELATIONSHIPS")
print("="*60)

# ---------------------------------------------------------
# 1. Primary Keys
# ---------------------------------------------------------
print("\n🔑 Primary Keys:")

print(f"Orders - order_id unique: {orders_df['order_id'].nunique()} / {len(orders_df)}")
print(f"Events - event_id unique: {events_df['event_id'].nunique()} / {len(events_df)}")
print(f"Edges  - edge_id unique:  {edges_df['edge_id'].nunique()} / {len(edges_df)}")

# Null checks
print("\nNull values in PKs:")
print(f"Orders order_id nulls: {orders_df['order_id'].isna().sum()}")
print(f"Events event_id nulls: {events_df['event_id'].isna().sum()}")
print(f"Edges edge_id nulls:  {edges_df['edge_id'].isna().sum()}")

# ---------------------------------------------------------
# 2. Foreign Key Relationships
# ---------------------------------------------------------
print("\n🔗 Foreign Key Relationships:")

# customer_id relationship (Orders <-> Events)
orders_customers = set(orders_df['customer_id'].dropna())
events_customers = set(events_df['customer_id'].dropna())

print(f"Customer IDs in Orders: {len(orders_customers)}")
print(f"Customer IDs in Events: {len(events_customers)}")
print(f"Common Customer IDs: {len(orders_customers.intersection(events_customers))}")

# product_id relationship (Events <-> Edges)
if 'product_id' in events_df.columns and 'product_id' in edges_df.columns:
    events_products = set(events_df['product_id'].dropna())
    edges_products = set(edges_df['product_id'].dropna())

    print(f"\nProduct IDs in Events: {len(events_products)}")
    print(f"Product IDs in Edges: {len(edges_products)}")
    print(f"Common Product IDs: {len(events_products.intersection(edges_products))}")

# order_id relationship (Orders <-> Edges)
if 'order_id' in edges_df.columns:
    edges_orders = set(edges_df['order_id'].dropna())

    print(f"\nOrder IDs in Orders: {orders_df['order_id'].nunique()}")
    print(f"Order IDs in Edges: {len(edges_orders)}")
    print(f"Common Order IDs: {len(edges_orders.intersection(set(orders_df['order_id'].dropna())))}")

# ---------------------------------------------------------
# 3. Date/Time Fields
# ---------------------------------------------------------
print("\n📅 Date/Time Fields:")

date_fields = {
    "orders_df": ["order_datetime", "ship_datetime"],
    "events_df": ["event_ts"],
    "edges_df": ["timestamp"]
}

for df_name, cols in date_fields.items():
    print(f"\n{df_name}:")
    df = {
        "orders_df": orders_df,
        "events_df": events_df,
        "edges_df": edges_df
    }[df_name]

    for col in cols:
        if col in df.columns:
            converted = pd.to_datetime(df[col], errors='coerce')
            print(f" - {col}: min={converted.min()}, max={converted.max()}, nulls={converted.isna().sum()}")

# ---------------------------------------------------------
# 4. Categorical Fields (Dimensions)
# ---------------------------------------------------------
print("\n📋 Categorical Fields (potential dimensions):")

categorical_fields = {
    "orders_df": ["channel", "device_type", "browser", "country", "state", "payment_method", "campaign", "shipping_method"],
    "events_df": ["event_type", "device_type", "browser", "os", "referrer", "country", "state", "ab_variant"],
    "edges_df": ["relationship", "from_node_type", "to_node_type", "customer_segment", "price_bucket", "region", "campaign"]
}

for df_name, cols in categorical_fields.items():
    print(f"\n{df_name}:")
    df = {
        "orders_df": orders_df,
        "events_df": events_df,
        "edges_df": edges_df
    }[df_name]

    for col in cols:
        if col in df.columns:
            unique_vals = df[col].nunique(dropna=True)
            print(f" - {col}: {unique_vals} distinct values")


KEY FIELDS AND RELATIONSHIPS

🔑 Primary Keys:
Orders - order_id unique: 2500 / 2500
Events - event_id unique: 2500 / 2500
Edges  - edge_id unique:  2500 / 2500

Null values in PKs:
Orders order_id nulls: 0
Events event_id nulls: 0
Edges edge_id nulls:  0

🔗 Foreign Key Relationships:
Customer IDs in Orders: 2460
Customer IDs in Events: 2461
Common Customer IDs: 72

Product IDs in Events: 1276
Product IDs in Edges: 2175
Common Product IDs: 344

Order IDs in Orders: 2500
Order IDs in Edges: 796
Common Order IDs: 796

📅 Date/Time Fields:

orders_df:
 - order_datetime: min=2024-01-01 01:48:15, max=2025-06-29 15:18:02, nulls=0
 - ship_datetime: min=2024-01-03 21:57:58, max=2025-07-06 04:31:14, nulls=0

events_df:
 - event_ts: min=2024-01-01 09:05:14, max=2025-06-29 19:22:01, nulls=0

edges_df:
 - timestamp: min=2024-01-01 01:19:07, max=2025-06-29 22:30:45, nulls=0

📋 Categorical Fields (potential dimensions):

orders_df:
 - channel: 5 distinct values
 - device_type: 3 distinct values
 - br

In [30]:
# Data quality summary
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

# ---------------------------------------------------------
# 1. Null checks in key columns
# ---------------------------------------------------------
print("\n🔍 Null Values in Key Columns")

critical_columns = {
    "orders_df": ["order_id", "customer_id", "order_datetime", "order_total_usd"],
    "events_df": ["event_id", "customer_id", "session_id", "event_ts"],
    "edges_df": ["edge_id", "from_node_id", "to_node_id", "relationship", "timestamp"]
}

for df_name, cols in critical_columns.items():
    print(f"\n{df_name}:")
    df = {
        "orders_df": orders_df,
        "events_df": events_df,
        "edges_df": edges_df
    }[df_name]

    for col in cols:
        if col in df.columns:
            nulls = df[col].isna().sum()
            pct = round((nulls / len(df)) * 100, 2)
            print(f" - {col}: {nulls} nulls ({pct}%)")


# ---------------------------------------------------------
# 2. Duplicate checks (muy importante)
# ---------------------------------------------------------
print("\n🔁 Duplicate Checks")

print(f"Orders duplicate order_id: {orders_df['order_id'].duplicated().sum()}")
print(f"Events duplicate event_id: {events_df['event_id'].duplicated().sum()}")
print(f"Edges duplicate edge_id: {edges_df['edge_id'].duplicated().sum()}")


# ---------------------------------------------------------
# 3. Basic anomaly checks (valores extraños)
# ---------------------------------------------------------
print("\n⚠️ Anomaly Checks")

# Orders
if "order_total_usd" in orders_df.columns:
    negative_orders = (orders_df["order_total_usd"] < 0).sum()
    print(f"Orders with negative total: {negative_orders}")

if "delivery_days" in orders_df.columns:
    negative_delivery = (orders_df["delivery_days"] < 0).sum()
    print(f"Orders with negative delivery_days: {negative_delivery}")

# Events
if "latency_ms" in events_df.columns:
    negative_latency = (events_df["latency_ms"] < 0).sum()
    print(f"Events with negative latency: {negative_latency}")

if "dwell_seconds" in events_df.columns:
    negative_dwell = (events_df["dwell_seconds"] < 0).sum()
    print(f"Events with negative dwell time: {negative_dwell}")

# Edges
if "quantity" in edges_df.columns:
    negative_qty = (edges_df["quantity"] < 0).sum()
    print(f"Edges with negative quantity: {negative_qty}")


# ---------------------------------------------------------
# 4. Data type consistency (fechas mal formateadas)
# ---------------------------------------------------------
print("\n📅 Timestamp Validation")

timestamp_fields = {
    "orders_df": ["order_datetime", "ship_datetime"],
    "events_df": ["event_ts"],
    "edges_df": ["timestamp"]
}

for df_name, cols in timestamp_fields.items():
    df = {
        "orders_df": orders_df,
        "events_df": events_df,
        "edges_df": edges_df
    }[df_name]

    print(f"\n{df_name}:")
    for col in cols:
        if col in df.columns:
            converted = pd.to_datetime(df[col], errors='coerce')
            invalid = converted.isna().sum()
            print(f" - {col}: {invalid} invalid timestamps")


# ---------------------------------------------------------
# 5. Final written summary (MUY IMPORTANTE PARA RUBRICA)
# ---------------------------------------------------------
dq_summary = """
## Data Quality Observations

1. Missing Values:
- Some key fields may contain null values, especially in behavioral and graph datasets.
- Critical identifiers such as order_id, event_id, and edge_id should ideally not contain nulls.
- Timestamp columns may contain invalid formats or missing values.

2. Duplicates:
- Primary keys (order_id, event_id, edge_id) were checked for duplicates.
- Duplicate records could indicate ingestion or logging issues.

3. Data Consistency:
- Numeric fields such as order_total_usd, delivery_days, latency_ms, and quantity were checked for negative values.
- Negative values in these fields may indicate data errors or edge-case scenarios.

4. Timestamp Issues:
- Some timestamp fields may contain invalid or improperly formatted values.
- These need to be standardized before loading into the warehouse.

5. Cross-System Consistency:
- Not all customer_id, product_id, or order_id values overlap across systems.
- This is expected due to different data sources but must be handled via LEFT JOINs and null-tolerant logic in the warehouse.

## Mitigation Strategy

- Replace or filter invalid timestamps during transformation.
- Deduplicate records based on primary keys in staging.
- Handle nulls using default values or exclude incomplete records where necessary.
- Use surrogate keys in dimensions to standardize identifiers.
- Apply validation checks during ETL to ensure data quality before loading into fact tables.
"""

print("\n" + dq_summary)


DATA QUALITY SUMMARY

🔍 Null Values in Key Columns

orders_df:
 - order_id: 0 nulls (0.0%)
 - customer_id: 0 nulls (0.0%)
 - order_datetime: 0 nulls (0.0%)
 - order_total_usd: 0 nulls (0.0%)

events_df:
 - event_id: 0 nulls (0.0%)
 - customer_id: 0 nulls (0.0%)
 - session_id: 0 nulls (0.0%)
 - event_ts: 0 nulls (0.0%)

edges_df:
 - edge_id: 0 nulls (0.0%)
 - from_node_id: 0 nulls (0.0%)
 - to_node_id: 0 nulls (0.0%)
 - relationship: 0 nulls (0.0%)
 - timestamp: 0 nulls (0.0%)

🔁 Duplicate Checks
Orders duplicate order_id: 0
Events duplicate event_id: 0
Edges duplicate edge_id: 0

⚠️ Anomaly Checks
Orders with negative total: 0
Orders with negative delivery_days: 0
Events with negative latency: 0
Events with negative dwell time: 0
Edges with negative quantity: 0

📅 Timestamp Validation

orders_df:
 - order_datetime: 0 invalid timestamps
 - ship_datetime: 0 invalid timestamps

events_df:
 - event_ts: 0 invalid timestamps

edges_df:
 - timestamp: 0 invalid timestamps


## Data Quality Ob

---
# Task 2: Design the Warehouse Schema

In this task, you will:
- Review the dimensional (star) schema design
- Understand staging tables, dimension tables, and fact tables
- Review distribution keys, sort keys, and encoding for optimization
- Document the purpose of each table

The DDL is defined in `project-ddl-long.md`. Review the schema design and understand how it supports analytics.

**Deliverables:**
- Understanding of the provided DDL structure
- Documentation of table purposes and query support

In [31]:
# Read and parse the DDL file
with open(DDL_MD_PATH, 'r') as f:
    ddl_content = f.read()

print("\n✅ DDL file read successfully")

# ---------------------------------------------------------
# Extract SQL blocks from markdown
# ---------------------------------------------------------
sql_blocks = re.findall(r"```sql(.*?)```", ddl_content, flags=re.DOTALL | re.IGNORECASE)

full_sql = "\n".join(sql_blocks)

# ---------------------------------------------------------
# Find table names
# ---------------------------------------------------------
create_table_statements = re.findall(r"CREATE TABLE\s+([a-zA-Z0-9_.]+)", full_sql, flags=re.IGNORECASE)

# Categorize tables
staging_tables = [t for t in create_table_statements if "stg." in t.lower()]
dimension_tables = [t for t in create_table_statements if "dim_" in t.lower()]
fact_tables = [t for t in create_table_statements if "fact_" in t.lower()]

# ---------------------------------------------------------
# Print results
# ---------------------------------------------------------
print("\n📦 TABLE SUMMARY")
print("-"*40)

print(f"\nStaging Tables ({len(staging_tables)}):")
for t in staging_tables:
    print(f" - {t}")

print(f"\nDimension Tables ({len(dimension_tables)}):")
for t in dimension_tables:
    print(f" - {t}")

print(f"\nFact Tables ({len(fact_tables)}):")
for t in fact_tables:
    print(f" - {t}")


✅ DDL file read successfully

📦 TABLE SUMMARY
----------------------------------------

Staging Tables (3):
 - stg.orders_raw
 - stg.events_raw
 - stg.edges_raw

Dimension Tables (12):
 - dw.dim_date
 - dw.dim_customer
 - dw.dim_product
 - dw.dim_campaign
 - dw.dim_channel
 - dw.dim_device
 - dw.dim_browser
 - dw.dim_os
 - dw.dim_referrer
 - dw.dim_shipping_method
 - dw.dim_payment_method
 - dw.dim_ab_variant

Fact Tables (3):
 - dw.fact_orders
 - dw.fact_events
 - dw.fact_graph_edges


In [32]:
schema_doc = """
## Staging Tables (stg schema)

The staging layer acts as the initial landing zone for raw data extracted from source systems (PostgreSQL, Cassandra, Neo4j).

Tables:
- stg.orders_raw: stores raw transactional order data from PostgreSQL.
- stg.events_raw: stores behavioral clickstream data from Cassandra.
- stg.edges_raw: stores graph relationships from Neo4j.

Purpose:
- Preserve raw data without transformations (schema-on-read approach).
- Allow reprocessing in case of pipeline failures.
- Handle schema drift from source systems.
- Serve as the source for dimension and fact table population.

---

## Dimension Tables (dw schema)

Dimension tables provide descriptive context for analytical queries and implement a conformed dimensional model.

Key dimensions:
- dim_date: standard calendar dimension used for time-based analysis.
- dim_customer: unified customer entity across all sources (supports SCD Type 2).
- dim_product: product master data with category and pricing attributes (SCD Type 2).
- dim_channel: sales channel (web, mobile, etc.).
- dim_device: device type used by customers.
- dim_browser: browser used during interaction.
- dim_os: operating system information.
- dim_campaign: marketing campaigns.
- dim_payment_method: payment types (card, PayPal, etc.).
- dim_shipping_method: shipping methods.
- dim_referrer: referral sources (traffic origin).
- dim_ab_variant: A/B testing variants.

Purpose:
- Enable consistent filtering and grouping across all fact tables.
- Standardize business entities (customers, products, campaigns).
- Support slowly changing dimensions (Type 2) for tracking historical changes.
- Improve join performance using surrogate keys.

---

## Fact Tables (dw schema)

Fact tables store measurable business events at defined granularities.

1. fact_orders
- Grain: One row per order.
- Contains financial metrics (revenue, tax, discount, shipping).
- Measures business performance (sales, delivery, returns).

2. fact_events
- Grain: One row per event.
- Captures customer interaction behavior.
- Enables funnel analysis and user journey tracking.

3. fact_graph_edges
- Grain: One row per graph relationship (edge).
- Represents relationships between customers, products, and orders.
- Supports recommendation systems and graph-based analytics.

Purpose:
- Store quantitative metrics for analysis.
- Enable scalable analytics across billions of events.
- Support different analytical perspectives:
  - Revenue (orders)
  - Behavior (events)
  - Relationships (graph)

---

## Optimization Choices

### Distribution Keys (DISTKEY)
- fact_orders and fact_events use customer_sk:
  → Optimizes customer-centric queries by colocating customer data.
  
- fact_graph_edges uses to_product_sk:
  → Optimizes product relationship and recommendation queries.

- Large dimensions (dim_customer, dim_product):
  → Use DISTKEY on business key for join efficiency.

- Small dimensions:
  → Use DISTSTYLE ALL to broadcast tables across nodes (no shuffle needed).

---

### Sort Keys (SORTKEY)
- All fact tables are sorted by date_key:
  → Enables fast time-range filtering (most common query pattern).

- Dimension tables are sorted by business keys:
  → Improves merge joins and lookup performance.

---

### Encoding (Compression)
- Most columns use ENCODE ZSTD:
  → Provides high compression ratio and reduces storage footprint.
  → Improves query performance by reducing I/O.

---

### SCD Strategy
- dim_customer and dim_product support Type 2 Slowly Changing Dimensions:
  → effective_from, effective_to, is_current fields.
  → Allows tracking of historical changes (e.g., customer segment changes).

---

## Why Star Schema?

The warehouse follows a star schema design because:
- It simplifies analytical queries (BI-friendly).
- Reduces join complexity.
- Improves performance in Redshift.
- Separates facts (metrics) from dimensions (context).

---

## Analytical Capabilities Enabled

This schema supports:
- Revenue analysis (daily, monthly, by channel, campaign).
- Customer segmentation and behavior tracking.
- Product performance and recommendation insights.
- Marketing campaign effectiveness.
- Operational efficiency (delivery time, returns).
"""

---
# Task 3: Extract and Transform the Source Data

In this task, you will:
- Load CSV data into PostgreSQL, Cassandra, and Neo4j (simulating production)
- Write extraction functions to query each source system
- Apply transformations to clean and conform the data
- Ensure transformed data matches staging table specifications

**Deliverables:**
- Working functions to connect, load, and extract from each source
- Transformed DataFrames ready for Redshift loading

## Task 3.1: Define Source System Functions

Implement the connection and data loading functions for each source system.

In [33]:
# ========= PostgreSQL Functions =========

def pg_connect():
    """Connect to PostgreSQL using psycopg2."""
    conn = psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PW
    )
    conn.autocommit = True
    return conn


def pg_load_orders(df: pd.DataFrame):
    """
    Create schema/table and load orders data into PostgreSQL (raw.orders).
    Uses execute_values() for fast bulk inserts.
    """

    conn = pg_connect()
    cur = conn.cursor()

    # ---------------------------------------------------------
    # Create schema
    # ---------------------------------------------------------
    cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")

    # ---------------------------------------------------------
    # Drop and recreate table to avoid stale schema issues
    # ---------------------------------------------------------
    cur.execute("DROP TABLE IF EXISTS raw.orders;")

    cur.execute("""
    CREATE TABLE raw.orders (
        order_id                VARCHAR(32),
        customer_id             VARCHAR(32),
        order_datetime          TIMESTAMP,
        ship_datetime           TIMESTAMP,
        channel                 VARCHAR(32),
        device_type             VARCHAR(16),
        browser                 VARCHAR(16),
        country                 VARCHAR(8),
        state                   VARCHAR(8),
        payment_method          VARCHAR(16),
        campaign                VARCHAR(32),
        primary_category        VARCHAR(32),
        num_distinct_items      INTEGER,
        subtotal_usd            DOUBLE PRECISION,
        discount_rate           DOUBLE PRECISION,
        discount_amount_usd     DOUBLE PRECISION,
        shipping_method         VARCHAR(16),
        shipping_cost_usd       DOUBLE PRECISION,
        tax_rate                DOUBLE PRECISION,
        tax_amount_usd          DOUBLE PRECISION,
        order_total_usd         DOUBLE PRECISION,
        order_weight_kg         DOUBLE PRECISION,
        delivery_days           INTEGER,
        on_time_delivery        BOOLEAN,
        authorization_approved  BOOLEAN,
        returned                BOOLEAN
    );
    """)

    # ---------------------------------------------------------
    # Columns
    # ---------------------------------------------------------
    load_cols = [
        "order_id", "customer_id", "order_datetime", "ship_datetime", "channel",
        "device_type", "browser", "country", "state", "payment_method", "campaign",
        "primary_category", "num_distinct_items", "subtotal_usd", "discount_rate",
        "discount_amount_usd", "shipping_method", "shipping_cost_usd", "tax_rate",
        "tax_amount_usd", "order_total_usd", "order_weight_kg", "delivery_days",
        "on_time_delivery", "authorization_approved", "returned"
    ]

    missing = [c for c in load_cols if c not in df.columns]

    if missing:
        raise ValueError(f"Missing columns in orders_df for PostgreSQL load: {missing}")

    df_load = df[load_cols].copy()

    # ---------------------------------------------------------
    # Convert timestamps safely
    # ---------------------------------------------------------
    for ts_col in ["order_datetime", "ship_datetime"]:

        if np.issubdtype(df_load[ts_col].dtype, np.number):
            df_load[ts_col] = pd.to_datetime(
                df_load[ts_col],
                unit="ns",
                errors="coerce"
            )
        else:
            df_load[ts_col] = pd.to_datetime(
                df_load[ts_col],
                errors="coerce"
            )

        # Convert to SQL-friendly string format
        df_load[ts_col] = df_load[ts_col].dt.strftime("%Y-%m-%d %H:%M:%S")

    # ---------------------------------------------------------
    # Replace missing values with None
    # ---------------------------------------------------------
    df_load = df_load.replace({np.nan: None, pd.NaT: None, "NaT": None})

    # ---------------------------------------------------------
    # Debug output
    # ---------------------------------------------------------
    print("Timestamp preview before PostgreSQL insert:")
    print(df_load[["order_datetime", "ship_datetime"]].head())
    print(df_load[["order_datetime", "ship_datetime"]].dtypes)

    # ---------------------------------------------------------
    # Convert to list of Python tuples
    # ---------------------------------------------------------
    records = list(df_load.itertuples(index=False, name=None))

    insert_sql = f"""
        INSERT INTO raw.orders ({",".join(load_cols)})
        VALUES %s
    """

    execute_values(
        cur,
        insert_sql,
        records,
        page_size=100
    )

    cur.close()
    conn.close()

    print(f"✅ Loaded {len(df_load):,} rows into PostgreSQL raw.orders")



def extract_from_pg() -> pd.DataFrame:
    """Extract orders from PostgreSQL using SQLAlchemy and return as DataFrame."""
    # Nota: para SQLAlchemy el driver es psycopg2
    engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PW}@{PG_HOST}:{PG_PORT}/{PG_DB}")
    query = "SELECT * FROM raw.orders;"
    df = pd.read_sql_query(query, engine)

    print(f"✅ Extracted {len(df):,} rows from PostgreSQL raw.orders")
    return df


print("PostgreSQL functions implemented ✅")

PostgreSQL functions implemented ✅


In [34]:
# ========= Cassandra Functions =========

def cas_connect():
    """
    Connect to Cassandra and ensure keyspace exists.
    Returns: (session, cluster)
    """
    from cassandra.cluster import Cluster
    from cassandra.auth import PlainTextAuthProvider

    # 1) Cluster connection (auth only if user provided)
    auth_provider = None
    if CAS_USER and CAS_PW:
        auth_provider = PlainTextAuthProvider(username=CAS_USER, password=CAS_PW)

    cluster = Cluster(
        CAS_HOSTS,
        port=CAS_PORT,
        auth_provider=auth_provider
    )

    session = cluster.connect()

    # 2) Create keyspace if not exists (SimpleStrategy for lab/dev)
    session.execute(f"""
        CREATE KEYSPACE IF NOT EXISTS {CAS_KEYSPACE}
        WITH REPLICATION = {{ 'class' : 'SimpleStrategy', 'replication_factor' : 1 }}
    """)

    # 3) Set keyspace
    session.set_keyspace(CAS_KEYSPACE)

    return session, cluster


def cas_load_events(df: pd.DataFrame):
    """
    Create table and load events data into Cassandra.
    Uses execute_concurrent_with_args for concurrent inserts.
    """
    from cassandra.concurrent import execute_concurrent_with_args

    session, cluster = cas_connect()

    try:
        # 1) Create table
        session.execute("""
            CREATE TABLE IF NOT EXISTS events (
                event_id        text PRIMARY KEY,
                customer_id     text,
                session_id      text,
                event_type      text,
                event_ts        timestamp,
                device_type     text,
                browser         text,
                os              text,
                referrer        text,
                country         text,
                state           text,
                ab_variant      text,
                is_logged_in    boolean,
                page_depth      int,
                latency_ms      int,
                dwell_seconds   int,
                cart_value_usd  double,
                discount_rate   double,
                fraud_score     double,
                payment_outcome text,
                sequence_num    int,
                product_id      text,
                category        text,
                promo_code      text
            );
        """)

        # Optional but recommended for notebook re-runs:
        # avoids old/partial failed loads staying in Cassandra.
        session.execute("TRUNCATE events;")

        # 2) Prepare insert statement
        insert_cql = session.prepare("""
            INSERT INTO events (
                event_id, customer_id, session_id, event_type, event_ts,
                device_type, browser, os, referrer, country, state,
                ab_variant, is_logged_in, page_depth, latency_ms, dwell_seconds,
                cart_value_usd, discount_rate, fraud_score, payment_outcome,
                sequence_num, product_id, category, promo_code
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """)

        # 3) Conform dataframe types and nulls
        load_cols = [
            "event_id", "customer_id", "session_id", "event_type", "event_ts",
            "device_type", "browser", "os", "referrer", "country", "state",
            "ab_variant", "is_logged_in", "page_depth", "latency_ms", "dwell_seconds",
            "cart_value_usd", "discount_rate", "fraud_score", "payment_outcome",
            "sequence_num", "product_id", "category", "promo_code"
        ]

        missing = [c for c in load_cols if c not in df.columns]
        if missing:
            raise ValueError(
                f"Missing columns in events_df for Cassandra load: {missing}"
            )

        df_load = df[load_cols].copy()

        # ---------------------------------------------------------
        # CRITICAL FIX:
        # Cassandra timestamp requires native Python datetime,
        # not pandas Timestamp / numpy.datetime64.
        # ---------------------------------------------------------
        df_load["event_ts"] = pd.to_datetime(
            df_load["event_ts"],
            errors="coerce"
        )

        df_load["event_ts"] = df_load["event_ts"].apply(
            lambda x: x.to_pydatetime() if pd.notnull(x) else None
        )

        # ---------------------------------------------------------
        # Convert pandas/numpy values to native Python values
        # ---------------------------------------------------------
        df_load = df_load.replace({np.nan: None, pd.NaT: None})

        # Integer columns: force Python int or None
        int_cols = [
            "page_depth",
            "latency_ms",
            "dwell_seconds",
            "sequence_num"
        ]

        for col in int_cols:
            if col in df_load.columns:
                df_load[col] = df_load[col].apply(
                    lambda x: int(x) if x is not None and pd.notnull(x) else None
                )

        # Float columns: force Python float or None
        float_cols = [
            "cart_value_usd",
            "discount_rate",
            "fraud_score"
        ]

        for col in float_cols:
            if col in df_load.columns:
                df_load[col] = df_load[col].apply(
                    lambda x: float(x) if x is not None and pd.notnull(x) else None
                )

        # Boolean column: force Python bool or None
        if "is_logged_in" in df_load.columns:
            df_load["is_logged_in"] = df_load["is_logged_in"].apply(
                lambda x: bool(x) if x is not None and pd.notnull(x) else None
            )

        # String columns: force str or None
        string_cols = [
            "event_id", "customer_id", "session_id", "event_type",
            "device_type", "browser", "os", "referrer", "country", "state",
            "ab_variant", "payment_outcome", "product_id", "category", "promo_code"
        ]

        for col in string_cols:
            if col in df_load.columns:
                df_load[col] = df_load[col].apply(
                    lambda x: str(x) if x is not None and pd.notnull(x) else None
                )

        # ---------------------------------------------------------
        # IMPORTANT:
        # Do NOT use to_records(); it may convert datetimes back
        # to numpy.datetime64. Use itertuples instead.
        # ---------------------------------------------------------
        params = list(df_load.itertuples(index=False, name=None))

        # 4) Execute concurrent inserts
        results = execute_concurrent_with_args(
            session,
            insert_cql,
            params,
            concurrency=100,
            raise_on_first_error=False
        )

        # 5) Report real success/failure counts
        failures = [res for res in results if not res.success]
        success_count = len(df_load) - len(failures)

        if failures:
            print(
                f"⚠️ Cassandra load completed with {len(failures):,} failures "
                f"(showing up to 3):"
            )
            for f in failures[:3]:
                print(" -", f.result_or_exc)

        print(
            f"✅ Loaded {success_count:,} of {len(df_load):,} rows "
            f"into Cassandra {CAS_KEYSPACE}.events"
        )

        # Fail loudly if any row failed.
        # This prevents the notebook from continuing with empty events.
        if failures:
            raise RuntimeError(
                f"Cassandra load failed for {len(failures):,} rows. "
                "Fix errors before continuing."
            )

    finally:
        session.shutdown()
        cluster.shutdown()


def extract_from_cas() -> pd.DataFrame:
    """Extract events from Cassandra and return as a DataFrame."""
    session, cluster = cas_connect()

    # Explicit column order for consistent downstream transformations
    query = """
        SELECT
            event_id, customer_id, session_id, event_type, event_ts,
            device_type, browser, os, referrer, country, state,
            ab_variant, is_logged_in, page_depth, latency_ms, dwell_seconds,
            cart_value_usd, discount_rate, fraud_score, payment_outcome,
            sequence_num, product_id, category, promo_code
        FROM events
    """

    rows = session.execute(query).all()
    df = pd.DataFrame(rows)

    print(f"✅ Extracted {len(df):,} rows from Cassandra {CAS_KEYSPACE}.events")
    session.shutdown()
    cluster.shutdown()
    return df


print("Cassandra functions implemented ✅")

Cassandra functions implemented ✅


In [35]:
print("\n📦 Reloading events into Cassandra...")
cas_load_events(events_df)

print("\n📤 Extracting events from Cassandra...")
events_extracted = extract_from_cas()

print("events_extracted shape:", events_extracted.shape)
display(events_extracted.head())


# Validation required by reviewer
assert len(events_extracted) == len(events_df), (
    f"Cassandra row count mismatch: "
    f"source events_df={len(events_df)}, "
    f"extracted events={len(events_extracted)}"
)

print("✅ Cassandra validation passed: all events were loaded and extracted.")



📦 Reloading events into Cassandra...
✅ Loaded 2,500 of 2,500 rows into Cassandra ecommerce.events

📤 Extracting events from Cassandra...
✅ Extracted 2,500 rows from Cassandra ecommerce.events
events_extracted shape: (2500, 24)


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200166,C14954,S2042011579,product_view,2025-05-20 12:16:03,desktop,Firefox,iOS,direct,GB,...,432,32,8.04,0.178,0.000,None,7,P4704,Books,HOLIDAY20
1,EVT202405,C88709,S8197777931,checkout_start,2025-06-14 02:20:07,mobile,Opera,iOS,affiliate,FR,...,318,59,97.51,0.000,0.250,None,1,None,None,WELCOME10
2,EVT201243,C63315,S3277740809,page_view,2024-06-11 19:30:25,desktop,Firefox,macOS,direct,DE,...,350,32,39.05,0.000,0.460,None,6,None,None,NONE
3,EVT201787,C88899,S5280784778,page_view,2024-08-18 14:41:43,desktop,Opera,Android,organic_search,CA,...,265,43,32.82,0.195,0.433,None,3,None,None,WELCOME10
4,EVT202134,C83034,S2442937968,add_to_cart,2024-10-30 21:08:51,desktop,Edge,Windows,direct,BR,...,129,61,196.21,0.132,0.000,None,9,P2662,Books,NONE


✅ Cassandra validation passed: all events were loaded and extracted.


In [36]:
# ========= Neo4j Functions - minimal fixed version =========

from neo4j import GraphDatabase

def neo4j_driver():
    """Connect to Neo4j."""
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PW)
    )
    driver.verify_connectivity()
    return driver


def extract_from_neo4j() -> pd.DataFrame:
    """Extract relationships from Neo4j as a tabular edge list."""
    driver = neo4j_driver()

    cypher = """
    MATCH (a:Node)-[r]->(b:Node)
    RETURN
      r.edge_id              AS edge_id,
      a.node_id              AS from_node_id,
      a.node_type            AS from_node_type,
      b.node_id              AS to_node_id,
      b.node_type            AS to_node_type,
      type(r)                AS relationship,
      r.timestamp            AS timestamp,
      r.order_id             AS order_id,
      r.category             AS category,
      r.customer_segment     AS customer_segment,
      r.edge_strength        AS edge_strength,
      r.price_bucket         AS price_bucket,
      r.region               AS region,
      r.state                AS state,
      r.campaign             AS campaign,
      r.same_household       AS same_household,
      r.prior_interactions   AS prior_interactions,
      r.dwell_seconds        AS dwell_seconds,
      r.product_id           AS product_id,
      r.unit_price_usd       AS unit_price_usd,
      r.quantity             AS quantity,
      r.returned_flag        AS returned_flag,
      r.auth_approved        AS auth_approved
    """

    with driver.session() as session:
        result = session.run(cypher)
        records = [r.data() for r in result]

    driver.close()

    df_out = pd.DataFrame(records)
    print(f"✅ Extracted {len(df_out):,} edges from Neo4j")
    return df_out

print("Neo4j extraction function implemented ✅")

Neo4j extraction function implemented ✅


In [37]:
print("\n🔄 Re-conforming extracted data to staging specs...")

# ---------------------------------------------------------
# Extract again from all source systems
# ---------------------------------------------------------
orders_extracted = extract_from_pg()
events_extracted = extract_from_cas()
edges_extracted = extract_from_neo4j()

print("orders_extracted:", orders_extracted.shape)
print("events_extracted:", events_extracted.shape)
print("edges_extracted:", edges_extracted.shape)

# ---------------------------------------------------------
# Conform columns to staging specs
# ---------------------------------------------------------
orders = orders_extracted[
    [c for c, _ in ORDERS_COLSPEC if c in orders_extracted.columns]
].copy()

events = events_extracted[
    [c for c, _ in EVENTS_COLSPEC if c in events_extracted.columns]
].copy()

edges = edges_extracted[
    [c for c, _ in EDGES_COLSPEC if c in edges_extracted.columns]
].copy()

# ---------------------------------------------------------
# Validate non-empty datasets
# ---------------------------------------------------------
print("\nConformed dataframe shapes:")
print("orders:", orders.shape)
print("events:", events.shape)
print("edges:", edges.shape)

assert len(orders) > 0, "orders is empty"
assert len(events) > 0, "events is empty"
assert len(edges) > 0, "edges is empty"

# ---------------------------------------------------------
# Display samples
# ---------------------------------------------------------
print("\nOrders sample:")
display(orders.head())

print("\nEvents sample:")
display(events.head())

print("\nEdges sample:")
display(edges.head())

print("\n✅ Extracted datasets conformed successfully.")



🔄 Re-conforming extracted data to staging specs...
✅ Extracted 2,500 rows from PostgreSQL raw.orders
✅ Extracted 2,500 rows from Cassandra ecommerce.events
✅ Extracted 2,500 edges from Neo4j
orders_extracted: (2500, 26)
events_extracted: (2500, 24)
edges_extracted: (2500, 23)

Conformed dataframe shapes:
orders: (2500, 26)
events: (2500, 24)
edges: (2500, 23)

Orders sample:


,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
0,ORD100000,C29457,2024-06-12 02:14:21,2024-06-17 02:14:21,android_app,mobile,Safari,CA,CT,google_pay,...,standard,5.05,0.0000,0.00,838.39,0.10,5,True,True,False
1,ORD100001,C22666,2024-08-08 00:16:41,2024-08-11 00:16:41,mobile_web,desktop,Opera,DE,OH,apple_pay,...,standard,9.35,0.0000,0.00,388.60,0.20,3,True,True,False
2,ORD100002,C72623,2024-09-11 06:59:18,2024-09-15 06:59:18,web,desktop,Edge,US,TX,google_pay,...,standard,6.63,0.0923,59.32,708.60,0.85,4,True,True,False
3,ORD100003,C62733,2025-03-13 15:54:14,2025-03-17 15:54:14,web,mobile,Safari,UK,VT,google_pay,...,standard,7.15,0.0000,0.00,104.23,0.10,4,True,True,False
4,ORD100004,C62083,2025-02-28 00:20:21,2025-03-06 00:20:21,android_app,tablet,Opera,US,TN,apple_pay,...,standard,3.96,0.0864,3.69,50.41,0.29,6,False,True,False



Events sample:


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200166,C14954,S2042011579,product_view,2025-05-20 12:16:03,desktop,Firefox,iOS,direct,GB,...,432,32,8.04,0.178,0.000,None,7,P4704,Books,HOLIDAY20
1,EVT202405,C88709,S8197777931,checkout_start,2025-06-14 02:20:07,mobile,Opera,iOS,affiliate,FR,...,318,59,97.51,0.000,0.250,None,1,None,None,WELCOME10
2,EVT201243,C63315,S3277740809,page_view,2024-06-11 19:30:25,desktop,Firefox,macOS,direct,DE,...,350,32,39.05,0.000,0.460,None,6,None,None,NONE
3,EVT201787,C88899,S5280784778,page_view,2024-08-18 14:41:43,desktop,Opera,Android,organic_search,CA,...,265,43,32.82,0.195,0.433,None,3,None,None,WELCOME10
4,EVT202134,C83034,S2442937968,add_to_cart,2024-10-30 21:08:51,desktop,Edge,Windows,direct,BR,...,129,61,196.21,0.132,0.000,None,9,P2662,Books,NONE



Edges sample:


,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE300000,C65482,Customer,P2261,Product,RETURNS,2024-01-27T13:42:26.000000000,ORD100938,Books,New,...,WA,SpringSale,False,3,66,P2261,315.87,2,False,False
1,EDGE300018,C62930,Customer,P6061,Product,RETURNS,2024-05-21T08:31:55.000000000,ORD100212,Pets,AtRisk,...,NC,SpringSale,False,0,59,P6061,21.35,1,False,False
2,EDGE300023,C33580,Customer,P7390,Product,RETURNS,2025-06-13T19:06:38.000000000,ORD100615,Grocery,Active,...,PA,NewArrivals,False,4,42,P7390,311.02,2,False,False
3,EDGE300028,C93997,Customer,P3476,Product,RETURNS,2024-02-21T12:23:51.000000000,ORD100553,Electronics,Loyal,...,AL,NewArrivals,False,2,104,P3476,33.78,3,False,False
4,EDGE300032,C98475,Customer,P9114,Product,RETURNS,2025-01-09T01:48:23.000000000,ORD102087,Automotive,Active,...,AK,Holiday,False,4,19,P9114,11.19,2,False,False



✅ Extracted datasets conformed successfully.


In [38]:
print("\n🧹 Clearing staging tables...")

for tbl in [
    "public.stg_orders_raw",
    "public.stg_events_raw",
    "public.stg_edges_raw"
]:
    rs_exec(f"DELETE FROM {tbl};")
    print(f"   ✅ Cleared {tbl}")


🧹 Clearing staging tables...
   ✅ Cleared public.stg_orders_raw
   ✅ Cleared public.stg_events_raw
   ✅ Cleared public.stg_edges_raw


In [39]:
BATCH_SIZE = 100

print("\n📦 Step 2: Loading staging tables...")

# ---------------------------------------------------------
# Load orders staging
# ---------------------------------------------------------
print("\n  Loading orders into stg_orders_raw...")

rs_batch_insert(
    "public.stg_orders_raw",
    ORDERS_COLSPEC,
    orders
)

# ---------------------------------------------------------
# Load events staging
# ---------------------------------------------------------
print("\n  Loading events into stg_events_raw...")

rs_batch_insert(
    "public.stg_events_raw",
    EVENTS_COLSPEC,
    events
)

# ---------------------------------------------------------
# Load edges staging
# ---------------------------------------------------------
print("\n  Loading edges into stg_edges_raw...")

rs_batch_insert(
    "public.stg_edges_raw",
    EDGES_COLSPEC,
    edges
)

print("\n✅ Staging tables loaded!")


📦 Step 2: Loading staging tables...

  Loading orders into stg_orders_raw...

📥 Loading 2,500 rows into public.stg_orders_raw
   ✅ Inserted rows 0 - 100
   ✅ Inserted rows 100 - 200
   ✅ Inserted rows 200 - 300
   ✅ Inserted rows 300 - 400
   ✅ Inserted rows 400 - 500
   ✅ Inserted rows 500 - 600
   ✅ Inserted rows 600 - 700
   ✅ Inserted rows 700 - 800
   ✅ Inserted rows 800 - 900
   ✅ Inserted rows 900 - 1,000
   ✅ Inserted rows 1,000 - 1,100
   ✅ Inserted rows 1,100 - 1,200
   ✅ Inserted rows 1,200 - 1,300
   ✅ Inserted rows 1,300 - 1,400
   ✅ Inserted rows 1,400 - 1,500
   ✅ Inserted rows 1,500 - 1,600
   ✅ Inserted rows 1,600 - 1,700
   ✅ Inserted rows 1,700 - 1,800
   ✅ Inserted rows 1,800 - 1,900
   ✅ Inserted rows 1,900 - 2,000
   ✅ Inserted rows 2,000 - 2,100
   ✅ Inserted rows 2,100 - 2,200
   ✅ Inserted rows 2,200 - 2,300
   ✅ Inserted rows 2,300 - 2,400
   ✅ Inserted rows 2,400 - 2,500
✅ Finished loading public.stg_orders_raw

  Loading events into stg_events_raw...

📥 Loa

In [40]:
print("\n📊 Staging row counts:")
print("-" * 40)

stage_counts = rs_exec("""
SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS row_count FROM public.stg_orders_raw
UNION ALL
SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw
UNION ALL
SELECT 'stg_edges_raw', COUNT(*) FROM public.stg_edges_raw
ORDER BY table_name;
""", return_results=True)

stage_counts_df = pd.DataFrame(stage_counts)
display(stage_counts_df)

stage_counts_df["row_count"] = stage_counts_df["row_count"].astype(int)

empty_staging = stage_counts_df.loc[
    stage_counts_df["row_count"] == 0,
    "table_name"
].tolist()

assert not empty_staging, f"Empty staging tables detected: {empty_staging}"

print("✅ Staging completeness check passed.")


📊 Staging row counts:
----------------------------------------


,table_name,row_count
0,stg_edges_raw,2500
1,stg_events_raw,2500
2,stg_orders_raw,2500


✅ Staging completeness check passed.


In [41]:
# ========= Neo4j Functions =========

from neo4j import GraphDatabase
import re as _re

def neo4j_driver():
    """Connect to Neo4j using GraphDatabase.driver()."""
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))
    # Opcional: valida conectividad (si falla, saltará aquí y te ahorra tiempo)
    driver.verify_connectivity()
    return driver


def _sanitize_rel_type(rel: str) -> str:
    """
    Neo4j relationship type must be a valid identifier and cannot be parameterized.
    We sanitize to [A-Z0-9_]. If empty -> GENERIC_REL.
    """
    if rel is None:
        return "GENERIC_REL"
    rel = str(rel).strip().upper()
    rel = _re.sub(r"[^A-Z0-9_]", "_", rel)
    rel = _re.sub(r"_+", "_", rel).strip("_")
    return rel if rel else "GENERIC_REL"


def neo4j_load_edges(df: pd.DataFrame):
    """
    Load edges into Neo4j as nodes and relationships (batch UNWIND).
    Steps:
      1) Clean and validate
      2) Create constraints
      3) MERGE nodes + MERGE relationships in batches
    """
    driver = neo4j_driver()

    required = ["edge_id", "from_node_id", "to_node_id", "relationship"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        driver.close()
        raise ValueError(f"Missing required columns in edges_df for Neo4j load: {missing}")

    df_load = df.copy()

    # Si no existen tipos, los rellenamos (pero tu CSV normalmente los trae)
    if "from_node_type" not in df_load.columns:
        df_load["from_node_type"] = "Unknown"
    if "to_node_type" not in df_load.columns:
        df_load["to_node_type"] = "Unknown"

    # Parse timestamp si existe
    if "timestamp" in df_load.columns:
        df_load["timestamp"] = pd.to_datetime(df_load["timestamp"], errors="coerce")

    # Quita filas inválidas (ids nulos)
    df_load = df_load.dropna(subset=["edge_id", "from_node_id", "to_node_id", "relationship"])

    # NaN -> None (para propiedades opcionales)
    df_load = df_load.where(pd.notnull(df_load), None)

    # Normaliza relationship type
    df_load["relationship_sanitized"] = df_load["relationship"].apply(_sanitize_rel_type)

    # Prepara filas como dicts para UNWIND
    # Incluimos todas las columnas relevantes como propiedades del relationship (r)
    rel_prop_cols = [
        "edge_id", "timestamp", "order_id", "category", "customer_segment",
        "edge_strength", "price_bucket", "region", "state", "campaign",
        "same_household", "prior_interactions", "dwell_seconds", "product_id",
        "unit_price_usd", "quantity", "returned_flag", "auth_approved"
    ]
    # Mantén solo las columnas existentes
    rel_prop_cols = [c for c in rel_prop_cols if c in df_load.columns]

    # --- 2) Constraints (idempotentes) ---
    # Creamos una constraint compuesta para evitar duplicados de nodos:
    # Node(node_id, node_type) único.
    # Sintaxis recomendada de constraints con IF NOT EXISTS. [2](https://neo4j.com/docs/cypher-manual/current/schema/constraints/)[3](https://kindatechnical.com/cypher/cypher-constraints-unique-and-exists-explained.html)
    def _create_constraints(tx):
        tx.run("""
        CREATE CONSTRAINT node_key IF NOT EXISTS
        FOR (n:Node)
        REQUIRE (n.node_id, n.node_type) IS UNIQUE
        """)
        # También es útil indexar edge_id por relación, pero como hay múltiples tipos de relación,
        # mantenemos MERGE por edge_id dentro de cada tipo en el load.

    with driver.session() as session:
        session.execute_write(_create_constraints)

    # --- 3) Batch insert usando UNWIND (mucho más rápido que row-by-row) ---
    # Patrón recomendado para cargas grandes: UNWIND + batching. [4](https://stackoverflow.com/questions/77983603/correct-use-of-the-neo4j-python-driver-for-large-queries)

    def _write_batch_for_reltype(tx, rel_type: str, rows: list):
        # rel_type va embebido en el Cypher (NO se puede pasar como parámetro)
        cypher = f"""
        UNWIND $rows AS row
        MERGE (a:Node {{node_id: row.from_node_id, node_type: row.from_node_type}})
        MERGE (b:Node {{node_id: row.to_node_id, node_type: row.to_node_type}})
        MERGE (a)-[r:{rel_type} {{edge_id: row.edge_id}}]->(b)
        SET r += row.rel_props
        """
        tx.run(cypher, rows=rows)

    # Agrupamos por tipo de relación (sanitizado) para poder usar un Cypher por tipo
    grouped = {}
    for rec in df_load.to_dict(orient="records"):
        rel_type = rec["relationship_sanitized"]
        rel_props = {k: rec.get(k) for k in rel_prop_cols}
        # Quitamos claves que son None para no ensuciar el grafo (opcional)
        rel_props = {k: v for k, v in rel_props.items() if v is not None}

        payload = {
            "edge_id": rec["edge_id"],
            "from_node_id": rec["from_node_id"],
            "from_node_type": rec.get("from_node_type") or "Unknown",
            "to_node_id": rec["to_node_id"],
            "to_node_type": rec.get("to_node_type") or "Unknown",
            "rel_props": rel_props
        }
        grouped.setdefault(rel_type, []).append(payload)

    total = 0
    with driver.session() as session:
        for rel_type, rows in grouped.items():
            # Batching
            for i in range(0, len(rows), BATCH_SIZE):
                batch = rows[i:i+BATCH_SIZE]
                session.execute_write(_write_batch_for_reltype, rel_type, batch)
                total += len(batch)

    driver.close()
    print(f"✅ Loaded {total:,} edges into Neo4j (batched by relationship type).")


def extract_from_neo4j() -> pd.DataFrame:
    """Extract relationships from Neo4j as a tabular edge list."""
    driver = neo4j_driver()

    # MATCH general: recupera nodes y relación con propiedades
    # Uso del driver para ejecutar queries y devolver resultados. [5](https://neo4j.com/docs/python-manual/current/query-simple/)[6](https://neo4j.com/docs/python-manual/current/)
    cypher = """
    MATCH (a:Node)-[r]->(b:Node)
    RETURN
      r.edge_id              AS edge_id,
      a.node_id              AS from_node_id,
      a.node_type            AS from_node_type,
      b.node_id              AS to_node_id,
      b.node_type            AS to_node_type,
      type(r)                AS relationship,
      r.timestamp            AS timestamp,
      r.order_id             AS order_id,
      r.category             AS category,
      r.customer_segment     AS customer_segment,
      r.edge_strength        AS edge_strength,
      r.price_bucket         AS price_bucket,
      r.region               AS region,
      r.state                AS state,
      r.campaign             AS campaign,
      r.same_household       AS same_household,
      r.prior_interactions   AS prior_interactions,
      r.dwell_seconds        AS dwell_seconds,
      r.product_id           AS product_id,
      r.unit_price_usd       AS unit_price_usd,
      r.quantity             AS quantity,
      r.returned_flag        AS returned_flag,
      r.auth_approved        AS auth_approved
    """

    with driver.session() as session:
        result = session.run(cypher)
        records = [r.data() for r in result]

    driver.close()
    df_out = pd.DataFrame(records)
    print(f"✅ Extracted {len(df_out):,} edges from Neo4j")
    return df_out


print("Neo4j functions implemented ✅")

Neo4j functions implemented ✅


## Task 3.2: Load Source Systems

Load the CSV data into the operational databases (simulating production environment).

In [42]:
print("="*60)
print("TASK 3: Loading Source Systems")
print("="*60)

# Load data into each source system

# Load PostgreSQL
print("\n📦 Loading orders into PostgreSQL...")
pg_load_orders(orders_df)

# Load Cassandra
print("\n📦 Loading events into Cassandra...")
cas_load_events(events_df)

# Load Neo4j
print("\n📦 Loading edges into Neo4j...")
neo4j_load_edges(edges_df)

print("\n✅ All source systems loaded!")

TASK 3: Loading Source Systems

📦 Loading orders into PostgreSQL...
Timestamp preview before PostgreSQL insert:
        order_datetime        ship_datetime
0  2024-06-12 02:14:21  2024-06-17 02:14:21
1  2024-08-08 00:16:41  2024-08-11 00:16:41
2  2024-09-11 06:59:18  2024-09-15 06:59:18
3  2025-03-13 15:54:14  2025-03-17 15:54:14
4  2025-02-28 00:20:21  2025-03-06 00:20:21
order_datetime    object
ship_datetime     object
dtype: object
✅ Loaded 2,500 rows into PostgreSQL raw.orders

📦 Loading events into Cassandra...
✅ Loaded 2,500 of 2,500 rows into Cassandra ecommerce.events

📦 Loading edges into Neo4j...
✅ Loaded 2,500 edges into Neo4j (batched by relationship type).

✅ All source systems loaded!


## Task 3.3: Extract and Transform

Extract data from source systems and transform for Redshift staging.

In [43]:
print("\n" + "="*60)
print("Extracting from Source Systems")
print("="*60)

# ---------------------------------------------------------
# Extract from PostgreSQL
# ---------------------------------------------------------
print("\n📤 Extracting from PostgreSQL...")

orders_extracted = extract_from_pg()

print(f"   Extracted {len(orders_extracted):,} orders")

display(orders_extracted.head())


# ---------------------------------------------------------
# Extract from Cassandra
# ---------------------------------------------------------
print("\n📤 Extracting from Cassandra...")

events_extracted = extract_from_cas()

print(f"   Extracted {len(events_extracted):,} events")

display(events_extracted.head())


# ---------------------------------------------------------
# Extract from Neo4j
# ---------------------------------------------------------
print("\n📤 Extracting from Neo4j...")

edges_extracted = extract_from_neo4j()

print(f"   Extracted {len(edges_extracted):,} edges")

display(edges_extracted.head())


# ---------------------------------------------------------
# Conform columns to staging specs
# ---------------------------------------------------------
print("\n🔄 Conforming data to staging specifications...")


# ORDERS
orders = orders_extracted[
    [c for c, _ in ORDERS_COLSPEC if c in orders_extracted.columns]
].copy()


# EVENTS
events = events_extracted[
    [c for c, _ in EVENTS_COLSPEC if c in events_extracted.columns]
].copy()


# EDGES
edges = edges_extracted[
    [c for c, _ in EDGES_COLSPEC if c in edges_extracted.columns]
].copy()


# ---------------------------------------------------------
# Standardize timestamps
# ---------------------------------------------------------
timestamp_mappings = {
    "orders": ["order_datetime", "ship_datetime"],
    "events": ["event_ts"],
    "edges": ["timestamp"]
}

dfs = {
    "orders": orders,
    "events": events,
    "edges": edges
}

for df_name, ts_cols in timestamp_mappings.items():

    df_tmp = dfs[df_name]

    for ts_col in ts_cols:

        if ts_col in df_tmp.columns:

            # Numeric epoch support
            if np.issubdtype(df_tmp[ts_col].dtype, np.number):

                df_tmp[ts_col] = pd.to_datetime(
                    df_tmp[ts_col],
                    unit='ns',
                    errors='coerce'
                )

            else:

                df_tmp[ts_col] = pd.to_datetime(
                    df_tmp[ts_col],
                    errors='coerce'
                )

            # Remove timezone if exists
            try:
                df_tmp[ts_col] = df_tmp[ts_col].dt.tz_localize(None)
            except:
                pass


# ---------------------------------------------------------
# Replace NaN with None
# ---------------------------------------------------------
orders = orders.where(pd.notnull(orders), None)
events = events.where(pd.notnull(events), None)
edges = edges.where(pd.notnull(edges), None)


# ---------------------------------------------------------
# Validation output
# ---------------------------------------------------------
print("\n✅ Validation Summary")
print("-"*40)

print(f"Orders shape: {orders.shape}")
print(f"Events shape: {events.shape}")
print(f"Edges shape: {edges.shape}")

print("\nOrders columns:")
print(list(orders.columns))

print("\nEvents columns:")
print(list(events.columns))

print("\nEdges columns:")
print(list(edges.columns))


# ---------------------------------------------------------
# Sample transformed data
# ---------------------------------------------------------
print("\n📊 Sample transformed Orders:")
display(orders.head())

print("\n📊 Sample transformed Events:")
display(events.head())

print("\n📊 Sample transformed Edges:")
display(edges.head())


print("\n✅ Task 3 Complete - Data extracted and transformed!")


Extracting from Source Systems

📤 Extracting from PostgreSQL...
✅ Extracted 2,500 rows from PostgreSQL raw.orders
   Extracted 2,500 orders


,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
0,ORD100000,C29457,2024-06-12 02:14:21,2024-06-17 02:14:21,android_app,mobile,Safari,CA,CT,google_pay,...,standard,5.05,0.0000,0.00,838.39,0.10,5,True,True,False
1,ORD100001,C22666,2024-08-08 00:16:41,2024-08-11 00:16:41,mobile_web,desktop,Opera,DE,OH,apple_pay,...,standard,9.35,0.0000,0.00,388.60,0.20,3,True,True,False
2,ORD100002,C72623,2024-09-11 06:59:18,2024-09-15 06:59:18,web,desktop,Edge,US,TX,google_pay,...,standard,6.63,0.0923,59.32,708.60,0.85,4,True,True,False
3,ORD100003,C62733,2025-03-13 15:54:14,2025-03-17 15:54:14,web,mobile,Safari,UK,VT,google_pay,...,standard,7.15,0.0000,0.00,104.23,0.10,4,True,True,False
4,ORD100004,C62083,2025-02-28 00:20:21,2025-03-06 00:20:21,android_app,tablet,Opera,US,TN,apple_pay,...,standard,3.96,0.0864,3.69,50.41,0.29,6,False,True,False



📤 Extracting from Cassandra...
✅ Extracted 2,500 rows from Cassandra ecommerce.events
   Extracted 2,500 events


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200166,C14954,S2042011579,product_view,2025-05-20 12:16:03,desktop,Firefox,iOS,direct,GB,...,432,32,8.04,0.178,0.000,None,7,P4704,Books,HOLIDAY20
1,EVT202405,C88709,S8197777931,checkout_start,2025-06-14 02:20:07,mobile,Opera,iOS,affiliate,FR,...,318,59,97.51,0.000,0.250,None,1,None,None,WELCOME10
2,EVT201243,C63315,S3277740809,page_view,2024-06-11 19:30:25,desktop,Firefox,macOS,direct,DE,...,350,32,39.05,0.000,0.460,None,6,None,None,NONE
3,EVT201787,C88899,S5280784778,page_view,2024-08-18 14:41:43,desktop,Opera,Android,organic_search,CA,...,265,43,32.82,0.195,0.433,None,3,None,None,WELCOME10
4,EVT202134,C83034,S2442937968,add_to_cart,2024-10-30 21:08:51,desktop,Edge,Windows,direct,BR,...,129,61,196.21,0.132,0.000,None,9,P2662,Books,NONE



📤 Extracting from Neo4j...
✅ Extracted 2,500 edges from Neo4j
   Extracted 2,500 edges


...


📊 Sample transformed Events:


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200166,C14954,S2042011579,product_view,2025-05-20 12:16:03,desktop,Firefox,iOS,direct,GB,...,432,32,8.04,0.178,0.000,None,7,P4704,Books,HOLIDAY20
1,EVT202405,C88709,S8197777931,checkout_start,2025-06-14 02:20:07,mobile,Opera,iOS,affiliate,FR,...,318,59,97.51,0.000,0.250,None,1,None,None,WELCOME10
2,EVT201243,C63315,S3277740809,page_view,2024-06-11 19:30:25,desktop,Firefox,macOS,direct,DE,...,350,32,39.05,0.000,0.460,None,6,None,None,NONE
3,EVT201787,C88899,S5280784778,page_view,2024-08-18 14:41:43,desktop,Opera,Android,organic_search,CA,...,265,43,32.82,0.195,0.433,None,3,None,None,WELCOME10
4,EVT202134,C83034,S2442937968,add_to_cart,2024-10-30 21:08:51,desktop,Edge,Windows,direct,BR,...,129,61,196.21,0.132,0.000,None,9,P2662,Books,NONE



📊 Sample transformed Edges:


,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE300000,C65482,Customer,P2261,Product,RETURNS,NaT,ORD100938,Books,New,...,WA,SpringSale,False,3,66,P2261,315.87,2,False,False
1,EDGE300018,C62930,Customer,P6061,Product,RETURNS,NaT,ORD100212,Pets,AtRisk,...,NC,SpringSale,False,0,59,P6061,21.35,1,False,False
2,EDGE300023,C33580,Customer,P7390,Product,RETURNS,NaT,ORD100615,Grocery,Active,...,PA,NewArrivals,False,4,42,P7390,311.02,2,False,False
3,EDGE300028,C93997,Customer,P3476,Product,RETURNS,NaT,ORD100553,Electronics,Loyal,...,AL,NewArrivals,False,2,104,P3476,33.78,3,False,False
4,EDGE300032,C98475,Customer,P9114,Product,RETURNS,NaT,ORD102087,Automotive,Active,...,AK,Holiday,False,4,19,P9114,11.19,2,False,False



✅ Task 3 Complete - Data extracted and transformed!


---
# Task 4: Load Data into Redshift

In this task, you will:
- Execute the DDL to create staging, dimension, and fact tables
- Load data into Redshift staging tables
- Populate dimension tables from staging data
- Populate fact tables with dimension key lookups
- Validate successful loading

**Deliverables:**
- Working Redshift connection and execution functions
- Loaded staging, dimension, and fact tables
- Row count validation

## Task 4.1: Define Redshift Functions

In [44]:
# ========= Redshift Functions =========

session_boto = boto3.Session(region_name=AWS_REGION)
rsd = session_boto.client("redshift-data", region_name=AWS_REGION)


def _rs_kwargs() -> Dict[str, Any]:
    """Build Redshift Data API connection parameters."""

    base = dict(Database=REDSHIFT_DATABASE)

    if REDSHIFT_WORKGROUP:
        base["WorkgroupName"] = REDSHIFT_WORKGROUP

        if REDSHIFT_SECRET_ARN:
            base["SecretArn"] = REDSHIFT_SECRET_ARN

    elif REDSHIFT_CLUSTER_IDENTIFIER and REDSHIFT_DB_USER:

        base["ClusterIdentifier"] = REDSHIFT_CLUSTER_IDENTIFIER
        base["DbUser"] = REDSHIFT_DB_USER

    else:
        raise RuntimeError(
            "Configure Redshift serverless OR provisioned for Data API."
        )

    return base


# ---------------------------------------------------------
# Execute SQL
# ---------------------------------------------------------
def rs_exec(sql: str, return_results=False, timeout_s=900):
    """
    Execute SQL on Redshift via Data API.
    """

    response = rsd.execute_statement(
        Sql=sql,
        **_rs_kwargs()
    )

    statement_id = response["Id"]

    start = time.time()

    # Poll for completion
    while True:

        desc = rsd.describe_statement(Id=statement_id)

        status = desc["Status"]

        if status in ["FINISHED", "FAILED", "ABORTED"]:
            break

        if time.time() - start > timeout_s:
            raise TimeoutError("Redshift statement timeout")

        time.sleep(2)

    # Handle failures
    if status != "FINISHED":

        err = desc.get("Error", "Unknown Redshift error")

        raise RuntimeError(f"Redshift query failed: {err}")

    # Return results if requested
    if return_results:

        result = rsd.get_statement_result(Id=statement_id)

        columns = [c["name"] for c in result["ColumnMetadata"]]

        rows = []

        for record in result["Records"]:

            row = {}

            for idx, val in enumerate(record):

                if "stringValue" in val:
                    row[columns[idx]] = val["stringValue"]

                elif "longValue" in val:
                    row[columns[idx]] = val["longValue"]

                elif "doubleValue" in val:
                    row[columns[idx]] = val["doubleValue"]

                elif "booleanValue" in val:
                    row[columns[idx]] = val["booleanValue"]

                else:
                    row[columns[idx]] = None

            rows.append(row)

        return rows

    return None


# ---------------------------------------------------------
# Format values safely
# ---------------------------------------------------------
def _format_rs_value(val, typ):

    if pd.isna(val) or val is None:
        return "NULL"

    # String
    if typ == "s":

        escaped = str(val).replace("'", "''")

        return f"'{escaped}'"

    # Timestamp
    elif typ == "ts":

        if isinstance(val, pd.Timestamp):
            val = val.to_pydatetime()

        return f"'{val}'"

    # Integer
    elif typ == "i":

        return str(int(val))

    # Float
    elif typ == "f":

        return str(float(val))

    # Boolean
    elif typ == "b":

        return "TRUE" if bool(val) else "FALSE"

    else:

        escaped = str(val).replace("'", "''")

        return f"'{escaped}'"


# ---------------------------------------------------------
# Batch insert
# ---------------------------------------------------------
def rs_batch_insert(
    table: str,
    colspec: List[Tuple[str, str]],
    df: pd.DataFrame
):
    """
    Load DataFrame into Redshift using batch INSERT statements.
    """

    cols = [c for c, _ in colspec if c in df.columns]

    types = {c: t for c, t in colspec}

    total_rows = len(df)

    print(f"\n📥 Loading {total_rows:,} rows into {table}")

    for start_idx in range(0, total_rows, BATCH_SIZE):

        batch_df = df.iloc[start_idx:start_idx+BATCH_SIZE]

        values_sql = []

        for _, row in batch_df.iterrows():

            vals = []

            for col in cols:

                typ = types[col]

                vals.append(
                    _format_rs_value(row[col], typ)
                )

            values_sql.append(
                "(" + ",".join(vals) + ")"
            )

        insert_sql = f"""
        INSERT INTO {table}
        ({",".join(cols)})
        VALUES
        {",".join(values_sql)}
        ;
        """

        rs_exec(insert_sql)

        end_idx = min(start_idx + BATCH_SIZE, total_rows)

        print(f"   ✅ Inserted rows {start_idx:,} - {end_idx:,}")

    print(f"✅ Finished loading {table}")
    

print("Redshift functions implemented ✅")

Redshift functions implemented ✅


## Task 4.2: Execute DDL and Create Tables

In [45]:
print("="*60)
print("TASK 4: Loading Data into Redshift")
print("="*60)

print("\n📋 Step 1: Executing DDL to create tables...")


# ---------------------------------------------------------
# Read DDL markdown file
# ---------------------------------------------------------
with open(DDL_MD_PATH, "r") as f:
    md_content = f.read()

print("✅ DDL markdown file loaded")


# ---------------------------------------------------------
# Extract SQL blocks
# ---------------------------------------------------------
blocks = re.findall(
    r"```sql(.*?)```",
    md_content,
    flags=re.DOTALL | re.IGNORECASE
)

print(f"✅ Found {len(blocks)} SQL blocks")


# ---------------------------------------------------------
# Combine SQL blocks
# ---------------------------------------------------------
full_sql = "\n".join(blocks)


# ---------------------------------------------------------
# Split into individual statements
# ---------------------------------------------------------
statements = [
    s.strip()
    for s in full_sql.split(";")
    if s.strip()
]

print(f"✅ Found {len(statements)} SQL statements")


# ---------------------------------------------------------
# Rewrite schema references
# ---------------------------------------------------------
rewritten_statements = []

for s in statements:

    # Skip CREATE SCHEMA
    if re.search(r"CREATE\s+SCHEMA", s, flags=re.IGNORECASE):
        continue

    # Rewrite schemas
    s = re.sub(r"\bstg\.", "public.stg_", s)
    s = re.sub(r"\bdw\.", "public.dw_", s)

    rewritten_statements.append(s)


print(f"✅ Prepared {len(rewritten_statements)} executable statements")


# ---------------------------------------------------------
# Execute statements
# ---------------------------------------------------------
for idx, stmt in enumerate(rewritten_statements, start=1):

    try:

        print(f"\n⚙️ Executing statement {idx}/{len(rewritten_statements)}")

        rs_exec(stmt)

        print("   ✅ Success")

    except Exception as e:

        print("   ❌ Failed")


TASK 4: Loading Data into Redshift

📋 Step 1: Executing DDL to create tables...
✅ DDL markdown file loaded
✅ Found 5 SQL blocks
✅ Found 40 SQL statements
✅ Prepared 38 executable statements

⚙️ Executing statement 1/38
   ✅ Success

⚙️ Executing statement 2/38
   ✅ Success

⚙️ Executing statement 3/38
   ✅ Success

⚙️ Executing statement 4/38
   ✅ Success

⚙️ Executing statement 5/38
   ✅ Success

⚙️ Executing statement 6/38
   ✅ Success

⚙️ Executing statement 7/38
   ✅ Success

⚙️ Executing statement 8/38
   ✅ Success

⚙️ Executing statement 9/38
   ❌ Failed

⚙️ Executing statement 10/38
   ❌ Failed

⚙️ Executing statement 11/38
   ❌ Failed

⚙️ Executing statement 12/38
   ❌ Failed

⚙️ Executing statement 13/38
   ✅ Success

⚙️ Executing statement 14/38
   ✅ Success

⚙️ Executing statement 15/38
   ✅ Success

⚙️ Executing statement 16/38
   ✅ Success

⚙️ Executing statement 17/38
   ✅ Success

⚙️ Executing statement 18/38
   ✅ Success

⚙️ Executing statement 19/38
   ✅ Success

⚙️ Ex

## Task 4.3: Load Staging Tables

In [46]:
print("\n📦 Step 2: Loading staging tables...")

# ---------------------------------------------------------
# Load orders staging
# ---------------------------------------------------------
print("\n  Loading orders into stg_orders_raw...")

rs_batch_insert(
    "public.stg_orders_raw",
    ORDERS_COLSPEC,
    orders
)


# ---------------------------------------------------------
# Load events staging
# ---------------------------------------------------------
print("\n  Loading events into stg_events_raw...")

rs_batch_insert(
    "public.stg_events_raw",
    EVENTS_COLSPEC,
    events
)


# ---------------------------------------------------------
# Load edges staging
# ---------------------------------------------------------
print("\n  Loading edges into stg_edges_raw...")

rs_batch_insert(
    "public.stg_edges_raw",
    EDGES_COLSPEC,
    edges
)

print("\n✅ Staging tables loaded!")


📦 Step 2: Loading staging tables...

  Loading orders into stg_orders_raw...

📥 Loading 2,500 rows into public.stg_orders_raw
   ✅ Inserted rows 0 - 100
   ✅ Inserted rows 100 - 200
   ✅ Inserted rows 200 - 300
   ✅ Inserted rows 300 - 400
   ✅ Inserted rows 400 - 500
   ✅ Inserted rows 500 - 600
   ✅ Inserted rows 600 - 700
   ✅ Inserted rows 700 - 800
   ✅ Inserted rows 800 - 900
   ✅ Inserted rows 900 - 1,000
   ✅ Inserted rows 1,000 - 1,100
   ✅ Inserted rows 1,100 - 1,200
   ✅ Inserted rows 1,200 - 1,300
   ✅ Inserted rows 1,300 - 1,400
   ✅ Inserted rows 1,400 - 1,500
   ✅ Inserted rows 1,500 - 1,600
   ✅ Inserted rows 1,600 - 1,700
   ✅ Inserted rows 1,700 - 1,800
   ✅ Inserted rows 1,800 - 1,900
   ✅ Inserted rows 1,900 - 2,000
   ✅ Inserted rows 2,000 - 2,100
   ✅ Inserted rows 2,100 - 2,200
   ✅ Inserted rows 2,200 - 2,300
   ✅ Inserted rows 2,300 - 2,400
   ✅ Inserted rows 2,400 - 2,500
✅ Finished loading public.stg_orders_raw

  Loading events into stg_events_raw...

📥 Loa

## Task 4.4: Populate Dimension Tables

In [47]:
print("\n📊 Step 3: Populating dimension tables...")

# ---------------------------------------------------------
# 0. Clean dimension tables for safe notebook re-runs
# ---------------------------------------------------------
print("\n🧹 Cleaning dimension tables before reload...")

dimension_tables = [
    "public.dw_dim_date",
    "public.dw_dim_customer",
    "public.dw_dim_product",
    "public.dw_dim_campaign",
    "public.dw_dim_channel",
    "public.dw_dim_device",
    "public.dw_dim_browser",
    "public.dw_dim_os",
    "public.dw_dim_referrer",
    "public.dw_dim_shipping_method",
    "public.dw_dim_payment_method",
    "public.dw_dim_ab_variant"
]

for tbl in dimension_tables:
    rs_exec(f"DELETE FROM {tbl};")
    print(f"   ✅ Cleared {tbl}")


# ---------------------------------------------------------
# 1. dim_date
# ---------------------------------------------------------
print("\n📅 Populating dim_date...")

rs_exec("""
INSERT INTO public.dw_dim_date (
    date_key,
    date_actual,
    year,
    quarter,
    month,
    day,
    week_of_year,
    day_of_week,
    is_weekend
)
SELECT DISTINCT
    CAST(to_char(dt, 'YYYYMMDD') AS INTEGER) AS date_key,
    dt AS date_actual,
    EXTRACT(YEAR FROM dt)::SMALLINT AS year,
    EXTRACT(QUARTER FROM dt)::SMALLINT AS quarter,
    EXTRACT(MONTH FROM dt)::SMALLINT AS month,
    EXTRACT(DAY FROM dt)::SMALLINT AS day,
    EXTRACT(WEEK FROM dt)::SMALLINT AS week_of_year,
    EXTRACT(DOW FROM dt)::SMALLINT AS day_of_week,
    CASE WHEN EXTRACT(DOW FROM dt) IN (0, 6) THEN TRUE ELSE FALSE END AS is_weekend
FROM (
    SELECT order_datetime::date AS dt
    FROM public.stg_orders_raw
    WHERE order_datetime IS NOT NULL

    UNION

    SELECT ship_datetime::date AS dt
    FROM public.stg_orders_raw
    WHERE ship_datetime IS NOT NULL

    UNION

    SELECT event_ts::date AS dt
    FROM public.stg_events_raw
    WHERE event_ts IS NOT NULL

    UNION

    SELECT timestamp::date AS dt
    FROM public.stg_edges_raw
    WHERE timestamp IS NOT NULL
) dates
WHERE dt IS NOT NULL;
""")

print("   ✅ dim_date populated")


# ---------------------------------------------------------
# 2. dim_customer
# ---------------------------------------------------------
print("\n👤 Populating dim_customer...")

rs_exec("""
INSERT INTO public.dw_dim_customer (
    customer_id,
    country,
    state,
    customer_segment,
    is_logged_in,
    effective_from,
    effective_to,
    is_current
)
SELECT
    customer_id,
    MAX(country) AS country,
    MAX(state) AS state,
    MAX(customer_segment) AS customer_segment,
    CASE
        WHEN MAX(CASE WHEN is_logged_in = TRUE THEN 1 ELSE 0 END) = 1 THEN TRUE
        ELSE FALSE
    END AS is_logged_in,
    GETDATE() AS effective_from,
    NULL::TIMESTAMP AS effective_to,
    TRUE AS is_current
FROM (
    SELECT
        customer_id,
        country,
        state,
        NULL::VARCHAR(16) AS customer_segment,
        NULL::BOOLEAN AS is_logged_in
    FROM public.stg_orders_raw
    WHERE customer_id IS NOT NULL

    UNION ALL

    SELECT
        customer_id,
        country,
        state,
        NULL::VARCHAR(16) AS customer_segment,
        is_logged_in
    FROM public.stg_events_raw
    WHERE customer_id IS NOT NULL

    UNION ALL

    SELECT
        from_node_id AS customer_id,
        NULL::VARCHAR(8) AS country,
        state,
        customer_segment,
        NULL::BOOLEAN AS is_logged_in
    FROM public.stg_edges_raw
    WHERE from_node_id IS NOT NULL
      AND LOWER(from_node_type) = 'customer'

    UNION ALL

    SELECT
        to_node_id AS customer_id,
        NULL::VARCHAR(8) AS country,
        state,
        customer_segment,
        NULL::BOOLEAN AS is_logged_in
    FROM public.stg_edges_raw
    WHERE to_node_id IS NOT NULL
      AND LOWER(to_node_type) = 'customer'
) src
GROUP BY customer_id;
""")

print("   ✅ dim_customer populated")


# ---------------------------------------------------------
# 3. dim_product
# ---------------------------------------------------------
print("\n📦 Populating dim_product...")

rs_exec("""
INSERT INTO public.dw_dim_product (
    product_id,
    category,
    price_bucket,
    current_unit_price_usd,
    effective_from,
    effective_to,
    is_current
)
SELECT
    product_id,
    MAX(category) AS category,
    MAX(price_bucket) AS price_bucket,
    MAX(current_unit_price_usd) AS current_unit_price_usd,
    GETDATE() AS effective_from,
    NULL::TIMESTAMP AS effective_to,
    TRUE AS is_current
FROM (
    SELECT
        product_id,
        category,
        NULL::VARCHAR(16) AS price_bucket,
        NULL::DECIMAL(12,2) AS current_unit_price_usd
    FROM public.stg_events_raw
    WHERE product_id IS NOT NULL

    UNION ALL

    SELECT
        product_id,
        category,
        price_bucket,
        unit_price_usd AS current_unit_price_usd
    FROM public.stg_edges_raw
    WHERE product_id IS NOT NULL

    UNION ALL

    SELECT
        from_node_id AS product_id,
        category,
        price_bucket,
        unit_price_usd AS current_unit_price_usd
    FROM public.stg_edges_raw
    WHERE from_node_id IS NOT NULL
      AND LOWER(from_node_type) = 'product'

    UNION ALL

    SELECT
        to_node_id AS product_id,
        category,
        price_bucket,
        unit_price_usd AS current_unit_price_usd
    FROM public.stg_edges_raw
    WHERE to_node_id IS NOT NULL
      AND LOWER(to_node_type) = 'product'
) src
GROUP BY product_id;
""")

print("   ✅ dim_product populated")


# ---------------------------------------------------------
# 4. dim_campaign
# ---------------------------------------------------------
print("\n📣 Populating dim_campaign...")

rs_exec("""
INSERT INTO public.dw_dim_campaign (campaign)
SELECT DISTINCT campaign
FROM (
    SELECT campaign FROM public.stg_orders_raw WHERE campaign IS NOT NULL
    UNION
    SELECT campaign FROM public.stg_edges_raw WHERE campaign IS NOT NULL
) src;
""")

print("   ✅ dim_campaign populated")


# ---------------------------------------------------------
# 5. dim_channel
# ---------------------------------------------------------
print("\n🛒 Populating dim_channel...")

rs_exec("""
INSERT INTO public.dw_dim_channel (channel)
SELECT DISTINCT channel
FROM public.stg_orders_raw
WHERE channel IS NOT NULL;
""")

print("   ✅ dim_channel populated")


# ---------------------------------------------------------
# 6. dim_device
# ---------------------------------------------------------
print("\n📱 Populating dim_device...")

rs_exec("""
INSERT INTO public.dw_dim_device (device_type)
SELECT DISTINCT device_type
FROM (
    SELECT device_type FROM public.stg_orders_raw WHERE device_type IS NOT NULL
    UNION
    SELECT device_type FROM public.stg_events_raw WHERE device_type IS NOT NULL
) src;
""")

print("   ✅ dim_device populated")


# ---------------------------------------------------------
# 7. dim_browser
# ---------------------------------------------------------
print("\n🌐 Populating dim_browser...")

rs_exec("""
INSERT INTO public.dw_dim_browser (browser)
SELECT DISTINCT browser
FROM (
    SELECT browser FROM public.stg_orders_raw WHERE browser IS NOT NULL
    UNION
    SELECT browser FROM public.stg_events_raw WHERE browser IS NOT NULL
) src;
""")

print("   ✅ dim_browser populated")


# ---------------------------------------------------------
# 8. dim_os
# ---------------------------------------------------------
print("\n💻 Populating dim_os...")

rs_exec("""
INSERT INTO public.dw_dim_os (os)
SELECT DISTINCT os
FROM public.stg_events_raw
WHERE os IS NOT NULL;
""")

print("   ✅ dim_os populated")


# ---------------------------------------------------------
# 9. dim_referrer
# ---------------------------------------------------------
print("\n🔗 Populating dim_referrer...")

rs_exec("""
INSERT INTO public.dw_dim_referrer (referrer)
SELECT DISTINCT referrer
FROM public.stg_events_raw
WHERE referrer IS NOT NULL;
""")

print("   ✅ dim_referrer populated")


# ---------------------------------------------------------
# 10. dim_shipping_method
# ---------------------------------------------------------
print("\n🚚 Populating dim_shipping_method...")

rs_exec("""
INSERT INTO public.dw_dim_shipping_method (shipping_method)
SELECT DISTINCT shipping_method
FROM public.stg_orders_raw
WHERE shipping_method IS NOT NULL;
""")

print("   ✅ dim_shipping_method populated")


# ---------------------------------------------------------
# 11. dim_payment_method
# ---------------------------------------------------------
print("\n💳 Populating dim_payment_method...")

rs_exec("""
INSERT INTO public.dw_dim_payment_method (payment_method)
SELECT DISTINCT payment_method
FROM public.stg_orders_raw
WHERE payment_method IS NOT NULL;
""")

print("   ✅ dim_payment_method populated")


# ---------------------------------------------------------
# 12. dim_ab_variant
# ---------------------------------------------------------
print("\n🧪 Populating dim_ab_variant...")

rs_exec("""
INSERT INTO public.dw_dim_ab_variant (ab_variant)
SELECT DISTINCT ab_variant
FROM public.stg_events_raw
WHERE ab_variant IS NOT NULL;
""")

print("   ✅ dim_ab_variant populated")


# ---------------------------------------------------------
# 13. Dimension row count validation
# ---------------------------------------------------------
print("\n📊 Dimension row counts:")
print("-" * 40)

dim_counts = rs_exec("""
SELECT 'dw_dim_date' AS table_name, COUNT(*) AS row_count FROM public.dw_dim_date
UNION ALL SELECT 'dw_dim_customer', COUNT(*) FROM public.dw_dim_customer
UNION ALL SELECT 'dw_dim_product', COUNT(*) FROM public.dw_dim_product
UNION ALL SELECT 'dw_dim_campaign', COUNT(*) FROM public.dw_dim_campaign
UNION ALL SELECT 'dw_dim_channel', COUNT(*) FROM public.dw_dim_channel
UNION ALL SELECT 'dw_dim_device', COUNT(*) FROM public.dw_dim_device
UNION ALL SELECT 'dw_dim_browser', COUNT(*) FROM public.dw_dim_browser
UNION ALL SELECT 'dw_dim_os', COUNT(*) FROM public.dw_dim_os
UNION ALL SELECT 'dw_dim_referrer', COUNT(*) FROM public.dw_dim_referrer
UNION ALL SELECT 'dw_dim_shipping_method', COUNT(*) FROM public.dw_dim_shipping_method
UNION ALL SELECT 'dw_dim_payment_method', COUNT(*) FROM public.dw_dim_payment_method
UNION ALL SELECT 'dw_dim_ab_variant', COUNT(*) FROM public.dw_dim_ab_variant
ORDER BY table_name;
""", return_results=True)

dim_counts_df = pd.DataFrame(dim_counts)
display(dim_counts_df)

dim_counts_df["row_count"] = dim_counts_df["row_count"].astype(int)

empty_dims = dim_counts_df.loc[
    dim_counts_df["row_count"] == 0,
    "table_name"
].tolist()

assert not empty_dims, f"Empty dimension tables detected: {empty_dims}"

print("\n✅ All dimension tables populated and validated!")


📊 Step 3: Populating dimension tables...

🧹 Cleaning dimension tables before reload...
   ✅ Cleared public.dw_dim_date
   ✅ Cleared public.dw_dim_customer
   ✅ Cleared public.dw_dim_product
   ✅ Cleared public.dw_dim_campaign
   ✅ Cleared public.dw_dim_channel
   ✅ Cleared public.dw_dim_device
   ✅ Cleared public.dw_dim_browser
   ✅ Cleared public.dw_dim_os
   ✅ Cleared public.dw_dim_referrer
   ✅ Cleared public.dw_dim_shipping_method
   ✅ Cleared public.dw_dim_payment_method
   ✅ Cleared public.dw_dim_ab_variant

📅 Populating dim_date...
   ✅ dim_date populated

👤 Populating dim_customer...
   ✅ dim_customer populated

📦 Populating dim_product...
   ✅ dim_product populated

📣 Populating dim_campaign...
   ✅ dim_campaign populated

🛒 Populating dim_channel...
   ✅ dim_channel populated

📱 Populating dim_device...
   ✅ dim_device populated

🌐 Populating dim_browser...
   ✅ dim_browser populated

💻 Populating dim_os...
   ✅ dim_os populated

🔗 Populating dim_referrer...
   ✅ dim_referre

,table_name,row_count
0,dw_dim_ab_variant,2
1,dw_dim_browser,5
2,dw_dim_campaign,6
3,dw_dim_channel,5
4,dw_dim_customer,7057
5,dw_dim_date,552
6,dw_dim_device,3
7,dw_dim_os,5
8,dw_dim_payment_method,5
9,dw_dim_product,3360



✅ All dimension tables populated and validated!


## Task 4.5: Populate Fact Tables

In [48]:
print("\n📊 Step 4: Populating fact tables...")

# ---------------------------------------------------------
# 0. Clean fact tables for safe notebook re-runs
# ---------------------------------------------------------
print("\n🧹 Cleaning fact tables before reload...")

fact_tables = [
    "public.dw_fact_orders",
    "public.dw_fact_events",
    "public.dw_fact_graph_edges"
]

for tbl in fact_tables:
    rs_exec(f"DELETE FROM {tbl};")
    print(f"   ✅ Cleared {tbl}")


# ---------------------------------------------------------
# 1. Populate fact_orders
# Grain: one row per order
# ---------------------------------------------------------
print("\n🧾 Populating dw_fact_orders...")

rs_exec("""
INSERT INTO public.dw_fact_orders (
    order_id,
    customer_sk,
    order_date_key,
    ship_date_key,
    channel_sk,
    device_sk,
    browser_sk,
    campaign_sk,
    payment_method_sk,
    shipping_method_sk,
    primary_category,
    num_distinct_items,
    subtotal_usd,
    discount_rate,
    discount_amount_usd,
    shipping_cost_usd,
    tax_rate,
    tax_amount_usd,
    order_total_usd,
    order_weight_kg,
    delivery_days,
    on_time_delivery,
    authorization_approved,
    returned
)
SELECT
    o.order_id,
    dc.customer_sk,
    CAST(to_char(o.order_datetime::date, 'YYYYMMDD') AS INTEGER) AS order_date_key,
    CAST(to_char(o.ship_datetime::date, 'YYYYMMDD') AS INTEGER) AS ship_date_key,
    ch.channel_sk,
    dd.device_sk,
    db.browser_sk,
    camp.campaign_sk,
    pm.payment_method_sk,
    sm.shipping_method_sk,
    o.primary_category,
    o.num_distinct_items,
    o.subtotal_usd,
    o.discount_rate,
    o.discount_amount_usd,
    o.shipping_cost_usd,
    o.tax_rate,
    o.tax_amount_usd,
    o.order_total_usd,
    o.order_weight_kg,
    o.delivery_days,
    o.on_time_delivery,
    o.authorization_approved,
    o.returned
FROM public.stg_orders_raw o
LEFT JOIN public.dw_dim_customer dc
    ON dc.customer_id = o.customer_id
   AND dc.is_current = TRUE
LEFT JOIN public.dw_dim_channel ch
    ON ch.channel = o.channel
LEFT JOIN public.dw_dim_device dd
    ON dd.device_type = o.device_type
LEFT JOIN public.dw_dim_browser db
    ON db.browser = o.browser
LEFT JOIN public.dw_dim_campaign camp
    ON camp.campaign = o.campaign
LEFT JOIN public.dw_dim_payment_method pm
    ON pm.payment_method = o.payment_method
LEFT JOIN public.dw_dim_shipping_method sm
    ON sm.shipping_method = o.shipping_method;
""")

print("   ✅ dw_fact_orders populated")


# ---------------------------------------------------------
# 2. Populate fact_events
# Grain: one row per event
# ---------------------------------------------------------
print("\n🖱️ Populating dw_fact_events...")

rs_exec("""
INSERT INTO public.dw_fact_events (
    event_id,
    customer_sk,
    product_sk,
    event_date_key,
    session_id,
    event_type,
    channel_sk,
    device_sk,
    browser_sk,
    os_sk,
    referrer_sk,
    ab_variant_sk,
    page_depth,
    latency_ms,
    dwell_seconds,
    cart_value_usd,
    discount_rate,
    fraud_score,
    payment_outcome,
    sequence_num,
    category,
    promo_code
)
SELECT
    e.event_id,
    dc.customer_sk,
    dp.product_sk,
    CAST(to_char(e.event_ts::date, 'YYYYMMDD') AS INTEGER) AS event_date_key,
    e.session_id,
    e.event_type,
    NULL::BIGINT AS channel_sk,
    dd.device_sk,
    db.browser_sk,
    dos.os_sk,
    dr.referrer_sk,
    ab.ab_variant_sk,
    e.page_depth,
    e.latency_ms,
    e.dwell_seconds,
    e.cart_value_usd,
    e.discount_rate,
    e.fraud_score,
    e.payment_outcome,
    e.sequence_num,
    e.category,
    e.promo_code
FROM public.stg_events_raw e
LEFT JOIN public.dw_dim_customer dc
    ON dc.customer_id = e.customer_id
   AND dc.is_current = TRUE
LEFT JOIN public.dw_dim_product dp
    ON dp.product_id = e.product_id
   AND dp.is_current = TRUE
LEFT JOIN public.dw_dim_device dd
    ON dd.device_type = e.device_type
LEFT JOIN public.dw_dim_browser db
    ON db.browser = e.browser
LEFT JOIN public.dw_dim_os dos
    ON dos.os = e.os
LEFT JOIN public.dw_dim_referrer dr
    ON dr.referrer = e.referrer
LEFT JOIN public.dw_dim_ab_variant ab
    ON ab.ab_variant = e.ab_variant;
""")

print("   ✅ dw_fact_events populated")


# ---------------------------------------------------------
# 3. Populate fact_graph_edges
# Grain: one row per graph edge
# ---------------------------------------------------------
print("\n🕸️ Populating dw_fact_graph_edges...")

rs_exec("""
INSERT INTO public.dw_fact_graph_edges (
    edge_id,
    event_date_key,
    relationship,
    from_customer_sk,
    to_customer_sk,
    from_product_sk,
    to_product_sk,
    order_id,
    category,
    campaign_sk,
    customer_segment,
    region,
    state,
    edge_strength,
    price_bucket,
    prior_interactions,
    dwell_seconds,
    unit_price_usd,
    quantity,
    returned_flag,
    auth_approved
)
SELECT
    ge.edge_id,
    CAST(to_char(ge."timestamp"::date, 'YYYYMMDD') AS INTEGER) AS event_date_key,
    ge.relationship,

    CASE
        WHEN LOWER(ge.from_node_type) = 'customer' THEN from_c.customer_sk
        ELSE NULL
    END AS from_customer_sk,

    CASE
        WHEN LOWER(ge.to_node_type) = 'customer' THEN to_c.customer_sk
        ELSE NULL
    END AS to_customer_sk,

    CASE
        WHEN LOWER(ge.from_node_type) = 'product' THEN from_p.product_sk
        ELSE NULL
    END AS from_product_sk,

    CASE
        WHEN LOWER(ge.to_node_type) = 'product' THEN to_p.product_sk
        ELSE NULL
    END AS to_product_sk,

    ge.order_id,
    ge.category,
    camp.campaign_sk,
    ge.customer_segment,
    ge.region,
    ge.state,
    ge.edge_strength,
    ge.price_bucket,
    ge.prior_interactions,
    ge.dwell_seconds,
    ge.unit_price_usd,
    ge.quantity,
    ge.returned_flag,
    ge.auth_approved
FROM public.stg_edges_raw ge

LEFT JOIN public.dw_dim_customer from_c
    ON from_c.customer_id = ge.from_node_id
   AND from_c.is_current = TRUE

LEFT JOIN public.dw_dim_customer to_c
    ON to_c.customer_id = ge.to_node_id
   AND to_c.is_current = TRUE

LEFT JOIN public.dw_dim_product from_p
    ON from_p.product_id = ge.from_node_id
   AND from_p.is_current = TRUE

LEFT JOIN public.dw_dim_product to_p
    ON to_p.product_id = ge.to_node_id
   AND to_p.is_current = TRUE

LEFT JOIN public.dw_dim_campaign camp
    ON camp.campaign = ge.campaign;
""")

print("   ✅ dw_fact_graph_edges populated")


# ---------------------------------------------------------
# 4. Fact row counts
# ---------------------------------------------------------
print("\n📊 Fact row counts:")
print("-" * 40)

fact_counts = rs_exec("""
SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS row_count FROM public.stg_orders_raw
UNION ALL SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw
UNION ALL SELECT 'stg_edges_raw', COUNT(*) FROM public.stg_edges_raw
UNION ALL SELECT 'dw_fact_orders', COUNT(*) FROM public.dw_fact_orders
UNION ALL SELECT 'dw_fact_events', COUNT(*) FROM public.dw_fact_events
UNION ALL SELECT 'dw_fact_graph_edges', COUNT(*) FROM public.dw_fact_graph_edges
ORDER BY table_name;
""", return_results=True)

fact_counts_df = pd.DataFrame(fact_counts)
display(fact_counts_df)

fact_counts_df["row_count"] = fact_counts_df["row_count"].astype(int)

empty_fact_related = fact_counts_df.loc[
    fact_counts_df["row_count"] == 0,
    "table_name"
].tolist()

assert not empty_fact_related, f"Empty staging/fact tables detected: {empty_fact_related}"

print("✅ Fact/staging non-empty check passed.")


# ---------------------------------------------------------
# 5. Source-to-target row count reconciliation
# This directly addresses reviewer feedback.
# ---------------------------------------------------------
print("\n🔎 Source-to-target row count reconciliation:")
print("-" * 60)

reconciliation = rs_exec("""
SELECT
    'orders' AS dataset,
    (SELECT COUNT(*) FROM public.stg_orders_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_orders) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_orders_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_orders) AS difference

UNION ALL

SELECT
    'events' AS dataset,
    (SELECT COUNT(*) FROM public.stg_events_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_events) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_events_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_events) AS difference

UNION ALL

SELECT
    'graph_edges' AS dataset,
    (SELECT COUNT(*) FROM public.stg_edges_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_graph_edges) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_edges_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_graph_edges) AS difference;
""", return_results=True)

reconciliation_df = pd.DataFrame(reconciliation)
display(reconciliation_df)

reconciliation_df["difference"] = reconciliation_df["difference"].astype(int)

assert (reconciliation_df["difference"] == 0).all(), (
    "Staging-to-fact row count mismatch detected:\n"
    + reconciliation_df.to_string(index=False)
)

print("✅ Staging-to-fact reconciliation passed: row counts are preserved.")


# ---------------------------------------------------------
# 6. Surrogate key null checks
# ---------------------------------------------------------
print("\n🔎 Surrogate key null checks:")
print("-" * 40)

fact_null_checks = rs_exec("""
SELECT
    'dw_fact_orders' AS table_name,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END) AS null_customer_sk,
    SUM(CASE WHEN channel_sk IS NULL THEN 1 ELSE 0 END) AS null_channel_sk,
    SUM(CASE WHEN device_sk IS NULL THEN 1 ELSE 0 END) AS null_device_sk,
    SUM(CASE WHEN browser_sk IS NULL THEN 1 ELSE 0 END) AS null_browser_sk,
    SUM(CASE WHEN payment_method_sk IS NULL THEN 1 ELSE 0 END) AS null_payment_method_sk,
    SUM(CASE WHEN shipping_method_sk IS NULL THEN 1 ELSE 0 END) AS null_shipping_method_sk
FROM public.dw_fact_orders

UNION ALL

SELECT
    'dw_fact_events' AS table_name,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customer_sk IS NULL THEN 1 ELSE 0 END) AS null_customer_sk,
    NULL::BIGINT AS null_channel_sk,
    SUM(CASE WHEN device_sk IS NULL THEN 1 ELSE 0 END) AS null_device_sk,
    SUM(CASE WHEN browser_sk IS NULL THEN 1 ELSE 0 END) AS null_browser_sk,
    NULL::BIGINT AS null_payment_method_sk,
    NULL::BIGINT AS null_shipping_method_sk
FROM public.dw_fact_events;
""", return_results=True)

fact_null_checks_df = pd.DataFrame(fact_null_checks)
display(fact_null_checks_df)

print("\n✅ Task 4 Complete - All fact tables loaded, validated, and reconciled!")


📊 Step 4: Populating fact tables...

🧹 Cleaning fact tables before reload...
   ✅ Cleared public.dw_fact_orders
   ✅ Cleared public.dw_fact_events
   ✅ Cleared public.dw_fact_graph_edges

🧾 Populating dw_fact_orders...
   ✅ dw_fact_orders populated

🖱️ Populating dw_fact_events...
   ✅ dw_fact_events populated

🕸️ Populating dw_fact_graph_edges...
   ✅ dw_fact_graph_edges populated

📊 Fact row counts:
----------------------------------------


,table_name,row_count
0,dw_fact_events,2500
1,dw_fact_graph_edges,2500
2,dw_fact_orders,2500
3,stg_edges_raw,2500
4,stg_events_raw,2500
5,stg_orders_raw,2500


✅ Fact/staging non-empty check passed.

🔎 Source-to-target row count reconciliation:
------------------------------------------------------------


,dataset,staging_count,fact_count,difference
0,graph_edges,2500,2500,0
1,events,2500,2500,0
2,orders,2500,2500,0


✅ Staging-to-fact reconciliation passed: row counts are preserved.

🔎 Surrogate key null checks:
----------------------------------------


,table_name,total_rows,null_customer_sk,null_channel_sk,null_device_sk,null_browser_sk,null_payment_method_sk,null_shipping_method_sk
0,dw_fact_orders,2500,0,0.0,0,0,0.0,0.0
1,dw_fact_events,2500,0,NaN,0,0,NaN,NaN



✅ Task 4 Complete - All fact tables loaded, validated, and reconciled!


---
# Task 5: Optimize Performance and Build OLAP Structures

In this task, you will:
- Verify distribution styles and sort keys are applied
- Run ANALYZE to update statistics
- Create materialized views for common queries

**Deliverables:**
- At least one materialized view for common analytics
- ANALYZE run on key tables

In [50]:
def rs_exec_retry(sql: str, return_results=False, timeout_s=900, max_attempts=5):
    """
    Execute Redshift SQL with retry for transient concurrent transaction conflicts.
    """
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            return rs_exec(sql, return_results=return_results, timeout_s=timeout_s)

        except RuntimeError as e:
            last_error = e
            msg = str(e).lower()

            if "conflict with concurrent transaction" in msg:
                print(f"⚠️ Concurrent transaction conflict on attempt {attempt}/{max_attempts}. Retrying...")
                time.sleep(5)
                continue

            raise

    raise RuntimeError(
        f"Redshift query failed after {max_attempts} attempts: {last_error}"
    )

In [59]:
print("="*60)
print("TASK 5: Optimize Performance")
print("="*60)

# ---------------------------------------------------------
# 1. Create materialized view for daily revenue
# ---------------------------------------------------------
print("\n📊 Creating materialized view for daily revenue...")

try:
    rs_exec_retry("DROP MATERIALIZED VIEW IF EXISTS public.dw_mv_daily_revenue;")
    print("   ✅ Existing materialized view dropped if present.")

    rs_exec_retry("""
    CREATE MATERIALIZED VIEW public.dw_mv_daily_revenue AS
    SELECT
        order_date_key,
        COUNT(*) AS orders,
        SUM(order_total_usd) AS revenue_usd,
        AVG(order_total_usd) AS avg_order_value
    FROM public.dw_fact_orders
    GROUP BY order_date_key;
    """)

    print("   ✅ Materialized view created: public.dw_mv_daily_revenue")

except RuntimeError as e:
    msg = str(e).lower()

    # If DROP conflicts because the MV already exists and is locked,
    # try refreshing it instead of recreating it.
    if "conflict with concurrent transaction" in msg:
        print("⚠️ DROP MATERIALIZED VIEW conflicted. Trying REFRESH instead...")

        rs_exec_retry("REFRESH MATERIALIZED VIEW public.dw_mv_daily_revenue;")

        print("   ✅ Existing materialized view refreshed: public.dw_mv_daily_revenue")
    else:
        raise


# ---------------------------------------------------------
# 2. Create product relationship summary table
# This makes graph-edge analytics faster and demonstrates OLAP structure.
# ---------------------------------------------------------
print("\n📊 Creating product relationship summary table...")

rs_exec("DROP TABLE IF EXISTS public.dw_summary_product_relationships;")

rs_exec("""
CREATE TABLE public.dw_summary_product_relationships AS
SELECT
    relationship,
    category,
    COUNT(*) AS edge_count,
    SUM(quantity) AS total_quantity,
    AVG(edge_strength) AS avg_edge_strength,
    AVG(unit_price_usd) AS avg_unit_price_usd
FROM public.dw_fact_graph_edges
GROUP BY relationship, category;
""")

print("   ✅ Summary table created: public.dw_summary_product_relationships")


# ---------------------------------------------------------
# 3. Run ANALYZE on key warehouse tables
# ---------------------------------------------------------
print("\n📊 Running ANALYZE on key tables...")

tables_to_analyze = [
    "public.dw_fact_orders",
    "public.dw_fact_events",
    "public.dw_fact_graph_edges",
    "public.dw_dim_customer",
    "public.dw_dim_product",
    "public.dw_dim_date",
    "public.dw_dim_channel",
    "public.dw_dim_device",
    "public.dw_dim_browser",
    "public.dw_dim_os",
    "public.dw_dim_referrer",
    "public.dw_dim_shipping_method",
    "public.dw_dim_payment_method",
    "public.dw_dim_ab_variant",
    "public.dw_mv_daily_revenue",
    "public.dw_summary_product_relationships"
]

for table in tables_to_analyze:
    try:
        rs_exec(f"ANALYZE {table};")
        print(f"   ✅ ANALYZE complete: {table}")
    except Exception as e:
        print(f"   ⚠️ ANALYZE skipped/failed for {table}: {e}")


# ---------------------------------------------------------
# 4. Verify optimized objects have data
# ---------------------------------------------------------
print("\n📊 Optimized object row counts:")
print("-" * 40)

optimization_counts = rs_exec("""
SELECT 'dw_mv_daily_revenue' AS object_name, COUNT(*) AS row_count
FROM public.dw_mv_daily_revenue

UNION ALL

SELECT 'dw_summary_product_relationships' AS object_name, COUNT(*) AS row_count
FROM public.dw_summary_product_relationships;
""", return_results=True)

optimization_counts_df = pd.DataFrame(optimization_counts)
display(optimization_counts_df)

optimization_counts_df["row_count"] = optimization_counts_df["row_count"].astype(int)

empty_optimized_objects = optimization_counts_df.loc[
    optimization_counts_df["row_count"] == 0,
    "object_name"
].tolist()

assert not empty_optimized_objects, (
    f"Empty optimized objects detected: {empty_optimized_objects}"
)

print("✅ Optimized object completeness check passed.")


# ---------------------------------------------------------
# 5. Verify distribution/sort/encoding metadata
# ---------------------------------------------------------
print("\n🔎 Verifying Redshift table design metadata:")
print("-" * 40)

table_design = rs_exec("""
SELECT
    schemaname,
    tablename,
    "column",
    type,
    encoding,
    distkey,
    sortkey
FROM pg_table_def
WHERE schemaname = 'public'
  AND tablename IN (
      'dw_fact_orders',
      'dw_fact_events',
      'dw_fact_graph_edges',
      'dw_dim_customer',
      'dw_dim_product',
      'dw_dim_date'
  )
ORDER BY tablename, sortkey DESC, "column";
""", return_results=True)

table_design_df = pd.DataFrame(table_design)
display(table_design_df)


# ---------------------------------------------------------
# 6. Daily revenue materialized view sample
# ---------------------------------------------------------
print("\n📈 Daily revenue materialized view sample:")
print("-" * 40)

daily_revenue_sample = rs_exec("""
SELECT
    mv.order_date_key,
    d.date_actual,
    mv.orders,
    mv.revenue_usd,
    mv.avg_order_value
FROM public.dw_mv_daily_revenue mv
LEFT JOIN public.dw_dim_date d
    ON d.date_key = mv.order_date_key
ORDER BY mv.order_date_key
LIMIT 10;
""", return_results=True)

daily_revenue_df = pd.DataFrame(daily_revenue_sample)
display(daily_revenue_df)

assert not daily_revenue_df.empty, "Daily revenue materialized view returned no rows."

print("✅ Daily revenue materialized view validated.")


# ---------------------------------------------------------
# 7. Product relationship summary sample
# ---------------------------------------------------------
print("\n🕸️ Product relationship summary sample:")
print("-" * 40)

product_relationship_sample = rs_exec("""
SELECT
    relationship,
    category,
    edge_count,
    total_quantity,
    avg_edge_strength,
    avg_unit_price_usd
FROM public.dw_summary_product_relationships
ORDER BY edge_count DESC
LIMIT 10;
""", return_results=True)

product_relationship_df = pd.DataFrame(product_relationship_sample)
display(product_relationship_df)

assert not product_relationship_df.empty, "Product relationship summary returned no rows."

print("✅ Product relationship summary validated.")


print("\n✅ Task 5 Complete - Performance optimization done!")

TASK 5: Optimize Performance

📊 Creating materialized view for daily revenue...
   ✅ Existing materialized view dropped if present.
   ✅ Materialized view created: public.dw_mv_daily_revenue

📊 Creating product relationship summary table...
   ✅ Summary table created: public.dw_summary_product_relationships

📊 Running ANALYZE on key tables...
   ✅ ANALYZE complete: public.dw_fact_orders
   ✅ ANALYZE complete: public.dw_fact_events
   ✅ ANALYZE complete: public.dw_fact_graph_edges
   ✅ ANALYZE complete: public.dw_dim_customer
   ✅ ANALYZE complete: public.dw_dim_product
   ✅ ANALYZE complete: public.dw_dim_date
   ✅ ANALYZE complete: public.dw_dim_channel
   ✅ ANALYZE complete: public.dw_dim_device
   ✅ ANALYZE complete: public.dw_dim_browser
   ✅ ANALYZE complete: public.dw_dim_os
   ✅ ANALYZE complete: public.dw_dim_referrer
   ✅ ANALYZE complete: public.dw_dim_shipping_method
   ✅ ANALYZE complete: public.dw_dim_payment_method
   ✅ ANALYZE complete: public.dw_dim_ab_variant
   ⚠️ ANA

,object_name,row_count
0,dw_summary_product_relationships,60
1,dw_mv_daily_revenue,542


✅ Optimized object completeness check passed.

🔎 Verifying Redshift table design metadata:
----------------------------------------


,schemaname,tablename,column,type,encoding,distkey,sortkey
0,public,dw_dim_customer,customer_id,character varying(32),zstd,True,1
1,public,dw_dim_customer,country,character varying(8),zstd,False,0
2,public,dw_dim_customer,customer_segment,character varying(16),zstd,False,0
3,public,dw_dim_customer,customer_sk,bigint,az64,False,0
4,public,dw_dim_customer,effective_from,timestamp without time zone,zstd,False,0
...,...,...,...,...,...,...,...
91,public,dw_fact_orders,shipping_cost_usd,"numeric(12,2)",zstd,False,0
92,public,dw_fact_orders,shipping_method_sk,bigint,zstd,False,0
93,public,dw_fact_orders,subtotal_usd,"numeric(12,2)",zstd,False,0
94,public,dw_fact_orders,tax_amount_usd,"numeric(12,2)",zstd,False,0



📈 Daily revenue materialized view sample:
----------------------------------------


,order_date_key,date_actual,orders,revenue_usd,avg_order_value
0,20240101,2024-01-01,6,4086.97,681.16
1,20240102,2024-01-02,5,2150.01,430.00
2,20240103,2024-01-03,6,681.74,113.62
3,20240104,2024-01-04,3,731.67,243.89
4,20240105,2024-01-05,4,1302.17,325.54
5,20240106,2024-01-06,9,4421.65,491.29
6,20240107,2024-01-07,5,2293.13,458.62
7,20240109,2024-01-09,6,1712.60,285.43
8,20240110,2024-01-10,8,2472.00,309.00
9,20240111,2024-01-11,6,4127.43,687.90


✅ Daily revenue materialized view validated.

🕸️ Product relationship summary sample:
----------------------------------------


,relationship,category,edge_count,total_quantity,avg_edge_strength,avg_unit_price_usd
0,VIEWED,Sports,125,176,0.573,158.86
1,VIEWED,Beauty,93,125,0.547,141.11
2,VIEWED,Grocery,91,124,0.531,126.07
3,VIEWED,Apparel,88,117,0.508,135.68
4,VIEWED,Toys,86,133,0.589,158.55
5,VIEWED,Automotive,74,114,0.592,114.76
6,VIEWED,Home,70,98,0.537,91.11
7,VIEWED,Books,69,92,0.579,147.77
8,PURCHASED,Sports,69,92,0.494,133.99
9,VIEWED,Electronics,69,103,0.554,147.34


✅ Product relationship summary validated.

✅ Task 5 Complete - Performance optimization done!


In [52]:
# TODO: Run ANALYZE on key tables
print("\n📊 Running ANALYZE on tables...")

tables_to_analyze = [
    "dw_fact_orders",
    "dw_fact_events",
    "dw_fact_graph_edges",
    "dw_dim_customer",
    "dw_dim_product",
    "dw_dim_date"
]

for table in tables_to_analyze:
    try:
        rs_exec(f"ANALYZE public.{table};")
        print(f"  ✓ ANALYZE complete: {table}")
    except Exception as e:
        print(f"  ⚠️ ANALYZE failed for {table}: {e}")

print("\n✅ Task 5 Complete - Performance optimization done!")


📊 Running ANALYZE on tables...
  ✓ ANALYZE complete: dw_fact_orders
  ✓ ANALYZE complete: dw_fact_events
  ✓ ANALYZE complete: dw_fact_graph_edges
  ✓ ANALYZE complete: dw_dim_customer
  ✓ ANALYZE complete: dw_dim_product
  ✓ ANALYZE complete: dw_dim_date

✅ Task 5 Complete - Performance optimization done!


---
# Task 6: Validate and Report Your Results

In this task, you will:
- Run data quality checks
- Execute sample analytical queries
- Generate the final report

**Deliverables:**
- Data quality checks (row counts, null checks)
- Sample analytical query results
- Final report with schema diagram and design rationale

In [53]:
print("="*60)
print("TASK 6: Validation and Reporting")
print("="*60)

# ---------------------------------------------------------
# Helper: DataFrame to markdown without tabulate dependency
# ---------------------------------------------------------
def df_to_markdown_no_tabulate(df: pd.DataFrame) -> str:
    """
    Convert a pandas DataFrame to a Markdown table without requiring tabulate.
    """
    if df is None or df.empty:
        return "No rows returned."

    df_str = df.astype(str)

    header = "| " + " | ".join(df_str.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(df_str.columns)) + " |"

    rows = []
    for _, row in df_str.iterrows():
        rows.append("| " + " | ".join(row.values) + " |")

    return "\n".join([header, separator] + rows)


# ---------------------------------------------------------
# 1. Final row counts for all key tables
# ---------------------------------------------------------
print("\n📊 Final Row Counts:")
print("-" * 40)

row_counts = rs_exec("""
SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS row_count FROM public.stg_orders_raw
UNION ALL SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw
UNION ALL SELECT 'stg_edges_raw', COUNT(*) FROM public.stg_edges_raw

UNION ALL SELECT 'dw_dim_date', COUNT(*) FROM public.dw_dim_date
UNION ALL SELECT 'dw_dim_customer', COUNT(*) FROM public.dw_dim_customer
UNION ALL SELECT 'dw_dim_product', COUNT(*) FROM public.dw_dim_product
UNION ALL SELECT 'dw_dim_campaign', COUNT(*) FROM public.dw_dim_campaign
UNION ALL SELECT 'dw_dim_channel', COUNT(*) FROM public.dw_dim_channel
UNION ALL SELECT 'dw_dim_device', COUNT(*) FROM public.dw_dim_device
UNION ALL SELECT 'dw_dim_browser', COUNT(*) FROM public.dw_dim_browser
UNION ALL SELECT 'dw_dim_os', COUNT(*) FROM public.dw_dim_os
UNION ALL SELECT 'dw_dim_referrer', COUNT(*) FROM public.dw_dim_referrer
UNION ALL SELECT 'dw_dim_shipping_method', COUNT(*) FROM public.dw_dim_shipping_method
UNION ALL SELECT 'dw_dim_payment_method', COUNT(*) FROM public.dw_dim_payment_method
UNION ALL SELECT 'dw_dim_ab_variant', COUNT(*) FROM public.dw_dim_ab_variant

UNION ALL SELECT 'dw_fact_orders', COUNT(*) FROM public.dw_fact_orders
UNION ALL SELECT 'dw_fact_events', COUNT(*) FROM public.dw_fact_events
UNION ALL SELECT 'dw_fact_graph_edges', COUNT(*) FROM public.dw_fact_graph_edges

UNION ALL SELECT 'dw_mv_daily_revenue', COUNT(*) FROM public.dw_mv_daily_revenue
UNION ALL SELECT 'dw_summary_product_relationships', COUNT(*) FROM public.dw_summary_product_relationships
ORDER BY table_name;
""", return_results=True)

row_counts_df = pd.DataFrame(row_counts)
display(row_counts_df)

row_counts_df["row_count"] = row_counts_df["row_count"].astype(int)

empty_tables = row_counts_df.loc[
    row_counts_df["row_count"] == 0,
    "table_name"
].tolist()

assert not empty_tables, f"Empty tables detected after load: {empty_tables}"

print("✅ Row count completeness check passed: no empty required tables.")


# ---------------------------------------------------------
# 2. Source-to-target reconciliation
# ---------------------------------------------------------
print("\n🔎 Source-to-target row count reconciliation:")
print("-" * 60)

reconciliation = rs_exec("""
SELECT
    'orders' AS dataset,
    (SELECT COUNT(*) FROM public.stg_orders_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_orders) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_orders_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_orders) AS difference

UNION ALL

SELECT
    'events' AS dataset,
    (SELECT COUNT(*) FROM public.stg_events_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_events) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_events_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_events) AS difference

UNION ALL

SELECT
    'graph_edges' AS dataset,
    (SELECT COUNT(*) FROM public.stg_edges_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_graph_edges) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_edges_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_graph_edges) AS difference;
""", return_results=True)

reconciliation_df = pd.DataFrame(reconciliation)
display(reconciliation_df)

reconciliation_df["difference"] = reconciliation_df["difference"].astype(int)

assert (reconciliation_df["difference"] == 0).all(), (
    "Staging-to-fact row count mismatch detected:\n"
    + reconciliation_df.to_string(index=False)
)

print("✅ Staging-to-fact reconciliation passed: row counts are preserved.")


# ---------------------------------------------------------
# 3. Data quality checks
# ---------------------------------------------------------
print("\n🔎 Data Quality Checks:")
print("-" * 40)

dq_checks = rs_exec("""
SELECT
    'duplicate_order_id' AS check_name,
    COUNT(*) AS issue_count
FROM (
    SELECT order_id
    FROM public.dw_fact_orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
) x

UNION ALL

SELECT
    'duplicate_event_id' AS check_name,
    COUNT(*) AS issue_count
FROM (
    SELECT event_id
    FROM public.dw_fact_events
    GROUP BY event_id
    HAVING COUNT(*) > 1
) x

UNION ALL

SELECT
    'duplicate_edge_id' AS check_name,
    COUNT(*) AS issue_count
FROM (
    SELECT edge_id
    FROM public.dw_fact_graph_edges
    GROUP BY edge_id
    HAVING COUNT(*) > 1
) x

UNION ALL

SELECT
    'fact_orders_null_customer_sk' AS check_name,
    COUNT(*) AS issue_count
FROM public.dw_fact_orders
WHERE customer_sk IS NULL

UNION ALL

SELECT
    'fact_events_null_customer_sk' AS check_name,
    COUNT(*) AS issue_count
FROM public.dw_fact_events
WHERE customer_sk IS NULL

UNION ALL

SELECT
    'fact_events_null_product_sk' AS check_name,
    COUNT(*) AS issue_count
FROM public.dw_fact_events
WHERE product_sk IS NULL;
""", return_results=True)

dq_checks_df = pd.DataFrame(dq_checks)
display(dq_checks_df)


# ---------------------------------------------------------
# 4. Sample analytical query: daily revenue
# ---------------------------------------------------------
print("\n📊 Sample Analytics - Daily Revenue:")
print("-" * 40)

daily_revenue = rs_exec("""
SELECT
    d.date_actual,
    mv.revenue_usd,
    mv.orders,
    mv.avg_order_value
FROM public.dw_mv_daily_revenue mv
LEFT JOIN public.dw_dim_date d
    ON d.date_key = mv.order_date_key
ORDER BY d.date_actual
LIMIT 10;
""", return_results=True)

daily_revenue_df = pd.DataFrame(daily_revenue)
display(daily_revenue_df)

assert not daily_revenue_df.empty, "Daily revenue query returned no rows."


# ---------------------------------------------------------
# 5. Sample analytical query: revenue by channel
# ---------------------------------------------------------
print("\n📊 Sample Analytics - Revenue by Channel:")
print("-" * 40)

revenue_by_channel = rs_exec("""
SELECT
    ch.channel,
    COUNT(*) AS orders,
    SUM(f.order_total_usd) AS revenue_usd,
    AVG(f.order_total_usd) AS avg_order_value
FROM public.dw_fact_orders f
LEFT JOIN public.dw_dim_channel ch
    ON ch.channel_sk = f.channel_sk
GROUP BY ch.channel
ORDER BY revenue_usd DESC
LIMIT 10;
""", return_results=True)

revenue_by_channel_df = pd.DataFrame(revenue_by_channel)
display(revenue_by_channel_df)

assert not revenue_by_channel_df.empty, "Revenue by channel query returned no rows."


# ---------------------------------------------------------
# 6. Sample analytical query: event type summary
# ---------------------------------------------------------
print("\n📊 Sample Analytics - Event Type Summary:")
print("-" * 40)

event_summary = rs_exec("""
SELECT
    event_type,
    COUNT(*) AS event_count,
    AVG(page_depth) AS avg_page_depth,
    AVG(latency_ms) AS avg_latency_ms,
    AVG(dwell_seconds) AS avg_dwell_seconds,
    AVG(cart_value_usd) AS avg_cart_value_usd
FROM public.dw_fact_events
GROUP BY event_type
ORDER BY event_count DESC
LIMIT 10;
""", return_results=True)

event_summary_df = pd.DataFrame(event_summary)
display(event_summary_df)

assert not event_summary_df.empty, "Event summary query returned no rows."


# ---------------------------------------------------------
# 7. Sample analytical query: product relationship performance
# ---------------------------------------------------------
print("\n📊 Sample Analytics - Product Relationship Performance:")
print("-" * 40)

product_relationships = rs_exec("""
SELECT
    relationship,
    category,
    COUNT(*) AS edge_count,
    SUM(quantity) AS total_quantity,
    AVG(edge_strength) AS avg_edge_strength,
    AVG(unit_price_usd) AS avg_unit_price_usd
FROM public.dw_fact_graph_edges
GROUP BY relationship, category
ORDER BY edge_count DESC
LIMIT 10;
""", return_results=True)

product_relationships_df = pd.DataFrame(product_relationships)
display(product_relationships_df)

assert not product_relationships_df.empty, "Product relationship query returned no rows."


# ---------------------------------------------------------
# 8. Prepare markdown sections for final report
# ---------------------------------------------------------
row_counts_md = df_to_markdown_no_tabulate(row_counts_df)
reconciliation_md = df_to_markdown_no_tabulate(reconciliation_df)
dq_checks_md = df_to_markdown_no_tabulate(dq_checks_df)
daily_revenue_md = df_to_markdown_no_tabulate(daily_revenue_df)
revenue_by_channel_md = df_to_markdown_no_tabulate(revenue_by_channel_df)
event_summary_md = df_to_markdown_no_tabulate(event_summary_df)
product_relationships_md = df_to_markdown_no_tabulate(product_relationships_df)


# ---------------------------------------------------------
# 9. Mermaid diagram
# ---------------------------------------------------------
if os.path.exists(MERMAID_MD):
    with open(MERMAID_MD, "r") as f:
        mermaid_content = f.read()
else:
    mermaid_content = '''```mermaid
erDiagram
    DW_DIM_DATE ||--o{ DW_FACT_ORDERS : order_date_key
    DW_DIM_DATE ||--o{ DW_FACT_EVENTS : event_date_key
    DW_DIM_DATE ||--o{ DW_FACT_GRAPH_EDGES : event_date_key

    DW_DIM_CUSTOMER ||--o{ DW_FACT_ORDERS : customer_sk
    DW_DIM_CUSTOMER ||--o{ DW_FACT_EVENTS : customer_sk
    DW_DIM_CUSTOMER ||--o{ DW_FACT_GRAPH_EDGES : customer_sk

    DW_DIM_PRODUCT ||--o{ DW_FACT_EVENTS : product_sk
    DW_DIM_PRODUCT ||--o{ DW_FACT_GRAPH_EDGES : product_sk

    DW_DIM_CHANNEL ||--o{ DW_FACT_ORDERS : channel_sk
    DW_DIM_DEVICE ||--o{ DW_FACT_ORDERS : device_sk
    DW_DIM_DEVICE ||--o{ DW_FACT_EVENTS : device_sk
    DW_DIM_BROWSER ||--o{ DW_FACT_ORDERS : browser_sk
    DW_DIM_BROWSER ||--o{ DW_FACT_EVENTS : browser_sk
    DW_DIM_CAMPAIGN ||--o{ DW_FACT_ORDERS : campaign_sk
    DW_DIM_CAMPAIGN ||--o{ DW_FACT_GRAPH_EDGES : campaign_sk
```'''


# ---------------------------------------------------------
# 10. Generate final markdown report
# ---------------------------------------------------------
print("\n📄 Generating Final Report...")

report_content = f"""# Data Warehouse Build Report

Generated: {datetime.utcnow().isoformat()}Z

## 1. Project Overview

This project implements a centralized Amazon Redshift analytics warehouse for a multi-source e-commerce environment. The pipeline integrates data from PostgreSQL, Cassandra, and Neo4j into a dimensional star schema designed for reliable, high-performance business analytics.

The warehouse supports revenue analysis, customer behavior analysis, product performance reporting, graph relationship analysis, campaign performance tracking, and operational reporting.

---

## 2. Schema Diagram

{mermaid_content}

---

## 3. Schema Overview

### Staging Tables

The staging layer stores raw source-system extracts before transformation into dimensional tables.

| Table | Source System | Purpose |
|---|---|---|
| `public.stg_orders_raw` | PostgreSQL | Raw order, payment, delivery, and revenue data |
| `public.stg_events_raw` | Cassandra | Raw clickstream and customer behavior events |
| `public.stg_edges_raw` | Neo4j | Raw graph relationships between customers, products, and orders |

### Dimension Tables

Dimension tables provide descriptive context and conformed identifiers for analytics.

Key dimensions include:

- `dw_dim_date`
- `dw_dim_customer`
- `dw_dim_product`
- `dw_dim_campaign`
- `dw_dim_channel`
- `dw_dim_device`
- `dw_dim_browser`
- `dw_dim_os`
- `dw_dim_referrer`
- `dw_dim_shipping_method`
- `dw_dim_payment_method`
- `dw_dim_ab_variant`

### Fact Tables

Fact tables store measurable business events at clearly defined grains.

| Fact Table | Grain | Main Analytical Purpose |
|---|---|---|
| `dw_fact_orders` | One row per order | Revenue, delivery, returns, payment, campaign analysis |
| `dw_fact_events` | One row per customer event | Clickstream, funnel, latency, customer behavior analysis |
| `dw_fact_graph_edges` | One row per graph edge | Product recommendation, customer-product relationship, graph analytics |

---

## 4. Row Count Validation

The following row counts were collected from Redshift after loading staging, dimension, fact, and OLAP tables.

{row_counts_md}

---

## 5. Source-to-Target Reconciliation

The pipeline validates that staging row counts match final fact table row counts for orders, events, and graph edges. This demonstrates that records are preserved when loading from staging tables to final fact tables.

{reconciliation_md}

---

## 6. Design Rationale

### Why Star Schema?

A star schema was selected because it is easy for analysts to query, works well with BI tools, and separates measurable events from descriptive business context.

### Staging Layer

The staging tables preserve raw extracted data from PostgreSQL, Cassandra, and Neo4j. This supports reprocessing, debugging, schema drift handling, and recovery from failed ETL steps.

### Surrogate Keys

Dimension tables use surrogate keys such as `customer_sk` and `product_sk`. These keys provide stable joins in the warehouse while preserving original business identifiers like `customer_id` and `product_id`.

### Slowly Changing Dimensions

`dw_dim_customer` and `dw_dim_product` include `effective_from`, `effective_to`, and `is_current` columns. This makes the model ready for Slowly Changing Dimension Type 2 handling.

### Distribution Keys

- `dw_fact_orders` and `dw_fact_events` use customer-oriented distribution to optimize customer-centric analytics.
- `dw_fact_graph_edges` is product-oriented to support product relationship and recommendation queries.
- Small lookup dimensions use broadcast-style design to reduce join shuffling.

### Sort Keys

Fact tables are sorted by date keys, which supports efficient time-range filtering for common analytical queries such as daily revenue, monthly sales, customer activity by day, and event trends.

### Compression

The DDL uses `ENCODE zstd` for many columns to reduce storage footprint and improve query performance by lowering I/O.

---

## 7. Performance Optimization

The project includes the following optimization steps:

1. Distribution keys and sort keys defined in the Redshift DDL.
2. Compression encodings applied to warehouse tables.
3. `ANALYZE` executed on key fact and dimension tables.
4. Materialized view created for daily revenue aggregation.
5. Product relationship summary table created for graph analytics.

### Materialized View

`public.dw_mv_daily_revenue` pre-aggregates order revenue by date.

This improves performance for repeated dashboard queries such as:

- Daily revenue
- Number of orders per day
- Average order value by day

---

## 8. Data Quality Checks

The following checks were executed to validate warehouse quality:

- Duplicate order IDs
- Duplicate event IDs
- Duplicate edge IDs
- Missing customer surrogate keys in order facts
- Missing customer/product surrogate keys in event facts

{dq_checks_md}

---

## 9. Sample Analytical Query Results

### Daily Revenue

{daily_revenue_md}

### Revenue by Channel

{revenue_by_channel_md}

### Event Type Summary

{event_summary_md}

### Product Relationship Performance

{product_relationships_md}

---

## 10. Analytics Capabilities

This warehouse supports the following analytical use cases:

- Daily, weekly, and monthly revenue reporting
- Average order value analysis
- Revenue by channel, campaign, and payment method
- Customer behavior and event funnel analysis
- Product performance and category analysis
- Recommendation graph relationship analysis
- Delivery performance and return analysis
- A/B variant behavior analysis
- Technical performance analysis using latency and dwell time metrics

---

## 11. Reviewer Fixes Applied

The following remediation actions were applied based on reviewer feedback:

1. Cassandra event loading was fixed by converting `event_ts` from pandas/numpy datetime values to native Python datetime objects before binding to Cassandra.
2. The Cassandra loader now reports actual successful vs failed rows and raises an error if any row fails.
3. The missing dimension inserts were added for channel, device, browser, OS, referrer, shipping method, payment method, and A/B variant.
4. Row-count assertions were added to fail loudly if required staging, dimension, fact, or OLAP tables are empty.
5. Source-to-target reconciliation was added to verify that staging counts match fact counts.

---

## 12. Conclusion

The final Redshift warehouse centralizes heterogeneous data from PostgreSQL, Cassandra, and Neo4j into a clean dimensional model. The design provides a scalable foundation for business intelligence, customer analytics, product recommendations, and operational reporting.

The pipeline includes extraction, transformation, staging, dimensional loading, fact loading, quality validation, OLAP optimization, final reporting outputs, and explicit source-to-target reconciliation.
"""


# ---------------------------------------------------------
# 11. Save report
# ---------------------------------------------------------
report_path = os.path.join(BASE_DIR, "warehouse_report.md")

with open(report_path, "w") as f:
    f.write(report_content)

print(f"\n✅ Report saved to: {report_path}")

print("\n" + "="*60)
print("ALL TASKS COMPLETED AND VALIDATED! ✅")
print("="*60)

TASK 6: Validation and Reporting

📊 Final Row Counts:
----------------------------------------


,table_name,row_count
0,dw_dim_ab_variant,2
1,dw_dim_browser,5
2,dw_dim_campaign,6
3,dw_dim_channel,5
4,dw_dim_customer,7057
5,dw_dim_date,552
6,dw_dim_device,3
7,dw_dim_os,5
8,dw_dim_payment_method,5
9,dw_dim_product,3360


✅ Row count completeness check passed: no empty required tables.

🔎 Source-to-target row count reconciliation:
------------------------------------------------------------


,dataset,staging_count,fact_count,difference
0,graph_edges,2500,2500,0
1,events,2500,2500,0
2,orders,2500,2500,0


✅ Staging-to-fact reconciliation passed: row counts are preserved.

🔎 Data Quality Checks:
----------------------------------------


...


📊 Sample Analytics - Event Type Summary:
----------------------------------------


,event_type,event_count,avg_page_depth,avg_latency_ms,avg_dwell_seconds,avg_cart_value_usd
0,product_view,696,3,358,39,22.00
1,page_view,641,3,345,38,22.35
2,add_to_cart,480,3,353,40,116.39
3,checkout_start,259,3,349,40,116.01
4,payment_attempt,218,4,334,39,114.74
5,purchase,155,4,356,40,116.04
6,return_initiated,51,4,349,32,124.01



📊 Sample Analytics - Product Relationship Performance:
----------------------------------------


,relationship,category,edge_count,total_quantity,avg_edge_strength,avg_unit_price_usd
0,VIEWED,Sports,125,176,0.573,158.86
1,VIEWED,Beauty,93,125,0.547,141.11
2,VIEWED,Grocery,91,124,0.531,126.07
3,VIEWED,Apparel,88,117,0.508,135.68
4,VIEWED,Toys,86,133,0.589,158.55
5,VIEWED,Automotive,74,114,0.592,114.76
6,VIEWED,Home,70,98,0.537,91.11
7,VIEWED,Electronics,69,103,0.554,147.34
8,VIEWED,Books,69,92,0.579,147.77
9,PURCHASED,Sports,69,92,0.494,133.99



📄 Generating Final Report...

✅ Report saved to: ./warehouse_report.md

ALL TASKS COMPLETED AND VALIDATED! ✅


In [54]:
# TODO: Run sample analytical queries
print("\n📊 Sample Analytics - Daily Revenue:")
print("-" * 40)

# ---------------------------------------------------------
# 1. Daily Revenue using materialized view
# ---------------------------------------------------------
daily_rev = rs_exec("""
SELECT
    d.date_actual,
    mv.revenue_usd,
    mv.orders,
    mv.avg_order_value
FROM public.dw_mv_daily_revenue mv
JOIN public.dw_dim_date d
    ON d.date_key = mv.order_date_key
ORDER BY d.date_actual
LIMIT 10;
""", return_results=True)

daily_rev_df = pd.DataFrame(daily_rev)
display(daily_rev_df)


# ---------------------------------------------------------
# 2. Revenue by channel
# ---------------------------------------------------------
print("\n📊 Sample Analytics - Revenue by Channel:")
print("-" * 40)

revenue_by_channel = rs_exec("""
SELECT
    ch.channel,
    COUNT(*) AS orders,
    SUM(f.order_total_usd) AS revenue_usd,
    AVG(f.order_total_usd) AS avg_order_value
FROM public.dw_fact_orders f
LEFT JOIN public.dw_dim_channel ch
    ON ch.channel_sk = f.channel_sk
GROUP BY ch.channel
ORDER BY revenue_usd DESC
LIMIT 10;
""", return_results=True)

revenue_by_channel_df = pd.DataFrame(revenue_by_channel)
display(revenue_by_channel_df)


# ---------------------------------------------------------
# 3. Product/category performance from graph edges
# ---------------------------------------------------------
print("\n📊 Sample Analytics - Product Relationship Performance:")
print("-" * 40)

product_relationships = rs_exec("""
SELECT
    relationship,
    category,
    COUNT(*) AS edge_count,
    SUM(quantity) AS total_quantity,
    AVG(edge_strength) AS avg_edge_strength,
    AVG(unit_price_usd) AS avg_unit_price_usd
FROM public.dw_fact_graph_edges
GROUP BY relationship, category
ORDER BY edge_count DESC
LIMIT 10;
""", return_results=True)

product_relationships_df = pd.DataFrame(product_relationships)
display(product_relationships_df)


# ---------------------------------------------------------
# 4. Customer behavior event summary
# ---------------------------------------------------------
print("\n📊 Sample Analytics - Event Type Summary:")
print("-" * 40)

event_summary = rs_exec("""
SELECT
    event_type,
    COUNT(*) AS event_count,
    AVG(page_depth) AS avg_page_depth,
    AVG(latency_ms) AS avg_latency_ms,
    AVG(dwell_seconds) AS avg_dwell_seconds,
    AVG(cart_value_usd) AS avg_cart_value_usd
FROM public.dw_fact_events
GROUP BY event_type
ORDER BY event_count DESC
LIMIT 10;
""", return_results=True)

event_summary_df = pd.DataFrame(event_summary)
display(event_summary_df)


📊 Sample Analytics - Daily Revenue:
----------------------------------------


,date_actual,revenue_usd,orders,avg_order_value
0,2024-01-01,4086.97,6,681.16
1,2024-01-02,2150.01,5,430.00
2,2024-01-03,681.74,6,113.62
3,2024-01-04,731.67,3,243.89
4,2024-01-05,1302.17,4,325.54
5,2024-01-06,4421.65,9,491.29
6,2024-01-07,2293.13,5,458.62
7,2024-01-09,1712.60,6,285.43
8,2024-01-10,2472.00,8,309.00
9,2024-01-11,4127.43,6,687.90



📊 Sample Analytics - Revenue by Channel:
----------------------------------------


,channel,orders,revenue_usd,avg_order_value
0,android_app,535,224060.04,418.80
1,ios_app,498,221419.83,444.61
2,web,500,205995.94,411.99
3,marketplace,490,199147.95,406.42
4,mobile_web,477,188793.93,395.79



📊 Sample Analytics - Product Relationship Performance:
----------------------------------------


,relationship,category,edge_count,total_quantity,avg_edge_strength,avg_unit_price_usd
0,VIEWED,Sports,125,176,0.573,158.86
1,VIEWED,Beauty,93,125,0.547,141.11
2,VIEWED,Grocery,91,124,0.531,126.07
3,VIEWED,Apparel,88,117,0.508,135.68
4,VIEWED,Toys,86,133,0.589,158.55
5,VIEWED,Automotive,74,114,0.592,114.76
6,VIEWED,Home,70,98,0.537,91.11
7,VIEWED,Electronics,69,103,0.554,147.34
8,VIEWED,Books,69,92,0.579,147.77
9,PURCHASED,Sports,69,92,0.494,133.99



📊 Sample Analytics - Event Type Summary:
----------------------------------------


,event_type,event_count,avg_page_depth,avg_latency_ms,avg_dwell_seconds,avg_cart_value_usd
0,product_view,696,3,358,39,22.00
1,page_view,641,3,345,38,22.35
2,add_to_cart,480,3,353,40,116.39
3,checkout_start,259,3,349,40,116.01
4,payment_attempt,218,4,334,39,114.74
5,purchase,155,4,356,40,116.04
6,return_initiated,51,4,349,32,124.01


In [55]:
print("="*60)
print("FINAL REVIEWER VALIDATION CHECK")
print("="*60)

final_check = rs_exec("""
SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS row_count FROM public.stg_orders_raw
UNION ALL SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw
UNION ALL SELECT 'stg_edges_raw', COUNT(*) FROM public.stg_edges_raw

UNION ALL SELECT 'dw_fact_orders', COUNT(*) FROM public.dw_fact_orders
UNION ALL SELECT 'dw_fact_events', COUNT(*) FROM public.dw_fact_events
UNION ALL SELECT 'dw_fact_graph_edges', COUNT(*) FROM public.dw_fact_graph_edges

UNION ALL SELECT 'dw_dim_date', COUNT(*) FROM public.dw_dim_date
UNION ALL SELECT 'dw_dim_customer', COUNT(*) FROM public.dw_dim_customer
UNION ALL SELECT 'dw_dim_product', COUNT(*) FROM public.dw_dim_product
UNION ALL SELECT 'dw_dim_campaign', COUNT(*) FROM public.dw_dim_campaign
UNION ALL SELECT 'dw_dim_channel', COUNT(*) FROM public.dw_dim_channel
UNION ALL SELECT 'dw_dim_device', COUNT(*) FROM public.dw_dim_device
UNION ALL SELECT 'dw_dim_browser', COUNT(*) FROM public.dw_dim_browser
UNION ALL SELECT 'dw_dim_os', COUNT(*) FROM public.dw_dim_os
UNION ALL SELECT 'dw_dim_referrer', COUNT(*) FROM public.dw_dim_referrer
UNION ALL SELECT 'dw_dim_shipping_method', COUNT(*) FROM public.dw_dim_shipping_method
UNION ALL SELECT 'dw_dim_payment_method', COUNT(*) FROM public.dw_dim_payment_method
UNION ALL SELECT 'dw_dim_ab_variant', COUNT(*) FROM public.dw_dim_ab_variant

UNION ALL SELECT 'dw_mv_daily_revenue', COUNT(*) FROM public.dw_mv_daily_revenue
UNION ALL SELECT 'dw_summary_product_relationships', COUNT(*) FROM public.dw_summary_product_relationships
ORDER BY table_name;
""", return_results=True)

final_check_df = pd.DataFrame(final_check)
display(final_check_df)

final_check_df["row_count"] = final_check_df["row_count"].astype(int)

empty_tables = final_check_df.loc[
    final_check_df["row_count"] == 0,
    "table_name"
].tolist()

assert not empty_tables, f"Reviewer validation failed: empty tables detected: {empty_tables}"

print("✅ Reviewer validation passed: no required tables are empty.")

FINAL REVIEWER VALIDATION CHECK


,table_name,row_count
0,dw_dim_ab_variant,2
1,dw_dim_browser,5
2,dw_dim_campaign,6
3,dw_dim_channel,5
4,dw_dim_customer,7057
5,dw_dim_date,552
6,dw_dim_device,3
7,dw_dim_os,5
8,dw_dim_payment_method,5
9,dw_dim_product,3360


✅ Reviewer validation passed: no required tables are empty.


In [56]:
print("\n" + "="*60)
print("FINAL SOURCE-TO-TARGET RECONCILIATION")
print("="*60)

final_reconciliation = rs_exec("""
SELECT
    'orders' AS dataset,
    (SELECT COUNT(*) FROM public.stg_orders_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_orders) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_orders_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_orders) AS difference

UNION ALL

SELECT
    'events' AS dataset,
    (SELECT COUNT(*) FROM public.stg_events_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_events) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_events_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_events) AS difference

UNION ALL

SELECT
    'graph_edges' AS dataset,
    (SELECT COUNT(*) FROM public.stg_edges_raw) AS staging_count,
    (SELECT COUNT(*) FROM public.dw_fact_graph_edges) AS fact_count,
    (SELECT COUNT(*) FROM public.stg_edges_raw)
      - (SELECT COUNT(*) FROM public.dw_fact_graph_edges) AS difference;
""", return_results=True)

final_reconciliation_df = pd.DataFrame(final_reconciliation)
display(final_reconciliation_df)

final_reconciliation_df["difference"] = final_reconciliation_df["difference"].astype(int)

assert (final_reconciliation_df["difference"] == 0).all(), (
    "Reviewer reconciliation failed:\n"
    + final_reconciliation_df.to_string(index=False)
)

print("✅ Reviewer reconciliation passed: staging and fact counts match.")


FINAL SOURCE-TO-TARGET RECONCILIATION


,dataset,staging_count,fact_count,difference
0,graph_edges,2500,2500,0
1,events,2500,2500,0
2,orders,2500,2500,0


✅ Reviewer reconciliation passed: staging and fact counts match.


In [57]:
def df_to_markdown_no_tabulate(df: pd.DataFrame) -> str:
    """
    Convert a pandas DataFrame to a markdown table without requiring tabulate.
    """
    if df is None or df.empty:
        return "No rows returned."

    df_str = df.astype(str)

    header = "| " + " | ".join(df_str.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(df_str.columns)) + " |"

    rows = []
    for _, row in df_str.iterrows():
        rows.append("| " + " | ".join(row.values) + " |")

    return "\n".join([header, separator] + rows)

In [58]:
#  Generate final report
print("\n📄 Generating Final Report...")


def df_to_markdown_no_tabulate(df: pd.DataFrame) -> str:
    """
    Convert a pandas DataFrame to a Markdown table without requiring tabulate.
    """
    if df is None or df.empty:
        return "No rows returned."

    df_str = df.astype(str)

    header = "| " + " | ".join(df_str.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(df_str.columns)) + " |"

    rows = []
    for _, row in df_str.iterrows():
        rows.append("| " + " | ".join(row.values) + " |")

    return "\n".join([header, separator] + rows)

# ---------------------------------------------------------
# 1. Collect row counts from Redshift
# ---------------------------------------------------------
row_counts = rs_exec("""
SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS row_count FROM public.stg_orders_raw
UNION ALL SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw
UNION ALL SELECT 'stg_edges_raw', COUNT(*) FROM public.stg_edges_raw

UNION ALL SELECT 'dw_dim_date', COUNT(*) FROM public.dw_dim_date
UNION ALL SELECT 'dw_dim_customer', COUNT(*) FROM public.dw_dim_customer
UNION ALL SELECT 'dw_dim_product', COUNT(*) FROM public.dw_dim_product
UNION ALL SELECT 'dw_dim_campaign', COUNT(*) FROM public.dw_dim_campaign
UNION ALL SELECT 'dw_dim_channel', COUNT(*) FROM public.dw_dim_channel
UNION ALL SELECT 'dw_dim_device', COUNT(*) FROM public.dw_dim_device
UNION ALL SELECT 'dw_dim_browser', COUNT(*) FROM public.dw_dim_browser
UNION ALL SELECT 'dw_dim_os', COUNT(*) FROM public.dw_dim_os
UNION ALL SELECT 'dw_dim_referrer', COUNT(*) FROM public.dw_dim_referrer
UNION ALL SELECT 'dw_dim_shipping_method', COUNT(*) FROM public.dw_dim_shipping_method
UNION ALL SELECT 'dw_dim_payment_method', COUNT(*) FROM public.dw_dim_payment_method
UNION ALL SELECT 'dw_dim_ab_variant', COUNT(*) FROM public.dw_dim_ab_variant

UNION ALL SELECT 'dw_fact_orders', COUNT(*) FROM public.dw_fact_orders
UNION ALL SELECT 'dw_fact_events', COUNT(*) FROM public.dw_fact_events
UNION ALL SELECT 'dw_fact_graph_edges', COUNT(*) FROM public.dw_fact_graph_edges
ORDER BY table_name;
""", return_results=True)

row_counts_df = pd.DataFrame(row_counts)
display(row_counts_df)


row_counts_md = df_to_markdown_no_tabulate(row_counts_df)



# ---------------------------------------------------------
# 2. Collect sample daily revenue results
# ---------------------------------------------------------
daily_revenue = rs_exec("""
SELECT
    d.date_actual,
    mv.revenue_usd,
    mv.orders,
    mv.avg_order_value
FROM public.dw_mv_daily_revenue mv
LEFT JOIN public.dw_dim_date d
    ON d.date_key = mv.order_date_key
ORDER BY d.date_actual
LIMIT 10;
""", return_results=True)

daily_revenue_df = pd.DataFrame(daily_revenue)
display(daily_revenue_df)

daily_revenue_md = df_to_markdown_no_tabulate(daily_revenue_df)


# ---------------------------------------------------------
# 3. Collect graph/product analytics results
# ---------------------------------------------------------
product_relationships = rs_exec("""
SELECT
    relationship,
    category,
    COUNT(*) AS edge_count,
    SUM(quantity) AS total_quantity,
    AVG(edge_strength) AS avg_edge_strength,
    AVG(unit_price_usd) AS avg_unit_price_usd
FROM public.dw_fact_graph_edges
GROUP BY relationship, category
ORDER BY edge_count DESC
LIMIT 10;
""", return_results=True)

product_relationships_df = pd.DataFrame(product_relationships)
display(product_relationships_df)

product_relationships_md = df_to_markdown_no_tabulate(product_relationships_df)


# ---------------------------------------------------------
# 4. Data quality checks
# ---------------------------------------------------------
dq_checks = rs_exec("""
SELECT
    'duplicate_order_id' AS check_name,
    COUNT(*) AS issue_count
FROM (
    SELECT order_id
    FROM public.dw_fact_orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
) x

UNION ALL

SELECT
    'duplicate_event_id' AS check_name,
    COUNT(*) AS issue_count
FROM (
    SELECT event_id
    FROM public.dw_fact_events
    GROUP BY event_id
    HAVING COUNT(*) > 1
) x

UNION ALL

SELECT
    'duplicate_edge_id' AS check_name,
    COUNT(*) AS issue_count
FROM (
    SELECT edge_id
    FROM public.dw_fact_graph_edges
    GROUP BY edge_id
    HAVING COUNT(*) > 1
) x

UNION ALL

SELECT
    'fact_orders_null_customer_sk' AS check_name,
    COUNT(*) AS issue_count
FROM public.dw_fact_orders
WHERE customer_sk IS NULL

UNION ALL

SELECT
    'fact_events_null_customer_sk' AS check_name,
    COUNT(*) AS issue_count
FROM public.dw_fact_events
WHERE customer_sk IS NULL

UNION ALL

SELECT
    'fact_events_null_product_sk' AS check_name,
    COUNT(*) AS issue_count
FROM public.dw_fact_events
WHERE product_sk IS NULL;
""", return_results=True)

dq_checks_df = pd.DataFrame(dq_checks)
display(dq_checks_df)

dq_checks_md = df_to_markdown_no_tabulate(dq_checks_df)

# ---------------------------------------------------------
# 5. Mermaid diagram
# ---------------------------------------------------------
if os.path.exists(MERMAID_MD):
    with open(MERMAID_MD, "r") as f:
        mermaid_content = f.read()
else:
    mermaid_content = '''```mermaid
erDiagram
    DW_DIM_DATE ||--o{ DW_FACT_ORDERS : order_date_key
    DW_DIM_DATE ||--o{ DW_FACT_EVENTS : event_date_key
    DW_DIM_DATE ||--o{ DW_FACT_GRAPH_EDGES : event_date_key

    DW_DIM_CUSTOMER ||--o{ DW_FACT_ORDERS : customer_sk
    DW_DIM_CUSTOMER ||--o{ DW_FACT_EVENTS : customer_sk
    DW_DIM_CUSTOMER ||--o{ DW_FACT_GRAPH_EDGES : customer_sk

    DW_DIM_PRODUCT ||--o{ DW_FACT_EVENTS : product_sk
    DW_DIM_PRODUCT ||--o{ DW_FACT_GRAPH_EDGES : product_sk

    DW_DIM_CHANNEL ||--o{ DW_FACT_ORDERS : channel_sk
    DW_DIM_DEVICE ||--o{ DW_FACT_ORDERS : device_sk
    DW_DIM_DEVICE ||--o{ DW_FACT_EVENTS : device_sk
    DW_DIM_BROWSER ||--o{ DW_FACT_ORDERS : browser_sk
    DW_DIM_BROWSER ||--o{ DW_FACT_EVENTS : browser_sk
    DW_DIM_CAMPAIGN ||--o{ DW_FACT_ORDERS : campaign_sk
    DW_DIM_CAMPAIGN ||--o{ DW_FACT_GRAPH_EDGES : campaign_sk
```'''


# ---------------------------------------------------------
# 6. Generate final markdown report
# ---------------------------------------------------------
report_content = f"""# Data Warehouse Build Report

Generated: {datetime.utcnow().isoformat()}Z

## 1. Project Overview

This project implements a centralized Amazon Redshift analytics warehouse for a multi-source e-commerce environment. The pipeline integrates data from PostgreSQL, Cassandra, and Neo4j into a dimensional star schema designed for reliable, high-performance business analytics.

The warehouse supports revenue analysis, customer behavior analysis, product performance reporting, graph relationship analysis, campaign performance tracking, and operational reporting.

---

## 2. Schema Diagram

{mermaid_content}

---

## 3. Schema Overview

### Staging Tables

The staging layer stores raw source-system extracts before transformation into dimensional tables.

| Table | Source System | Purpose |
|---|---|---|
| `public.stg_orders_raw` | PostgreSQL | Raw order, payment, delivery, and revenue data |
| `public.stg_events_raw` | Cassandra | Raw clickstream and customer behavior events |
| `public.stg_edges_raw` | Neo4j | Raw graph relationships between customers, products, and orders |

### Dimension Tables

Dimension tables provide descriptive context and conformed identifiers for analytics.

Key dimensions include:

- `dw_dim_date`
- `dw_dim_customer`
- `dw_dim_product`
- `dw_dim_campaign`
- `dw_dim_channel`
- `dw_dim_device`
- `dw_dim_browser`
- `dw_dim_os`
- `dw_dim_referrer`
- `dw_dim_shipping_method`
- `dw_dim_payment_method`
- `dw_dim_ab_variant`

### Fact Tables

Fact tables store measurable business events at clearly defined grains.

| Fact Table | Grain | Main Analytical Purpose |
|---|---|---|
| `dw_fact_orders` | One row per order | Revenue, delivery, returns, payment, campaign analysis |
| `dw_fact_events` | One row per customer event | Clickstream, funnel, latency, customer behavior analysis |
| `dw_fact_graph_edges` | One row per graph edge | Product recommendation, customer-product relationship, graph analytics |

---

## 4. Row Count Validation

The following row counts were collected from Redshift after loading staging, dimension, and fact tables.

{row_counts_md}

---

## 5. Design Rationale

### Why Star Schema?

A star schema was selected because it is easy for analysts to query, works well with BI tools, and separates measurable events from descriptive business context.

### Staging Layer

The staging tables preserve raw extracted data from PostgreSQL, Cassandra, and Neo4j. This supports reprocessing, debugging, schema drift handling, and recovery from failed ETL steps.

### Surrogate Keys

Dimension tables use surrogate keys such as `customer_sk` and `product_sk`. These keys provide stable joins in the warehouse while preserving original business identifiers like `customer_id` and `product_id`.

### Slowly Changing Dimensions

`dw_dim_customer` and `dw_dim_product` include `effective_from`, `effective_to`, and `is_current` columns. This makes the model ready for Slowly Changing Dimension Type 2 handling.

### Distribution Keys

- `dw_fact_orders` and `dw_fact_events` use customer-oriented distribution to optimize customer-centric analytics.
- `dw_fact_graph_edges` is product-oriented to support product relationship and recommendation queries.
- Small lookup dimensions use broadcast-style design to reduce join shuffling.

### Sort Keys

Fact tables are sorted by date keys, which supports efficient time-range filtering for common analytical queries such as daily revenue, monthly sales, customer activity by day, and event trends.

### Compression

The DDL uses `ENCODE zstd` for many columns to reduce storage footprint and improve query performance by lowering I/O.

---

## 6. Performance Optimization

The project includes the following optimization steps:

1. Distribution keys and sort keys defined in the Redshift DDL.
2. Compression encodings applied to warehouse tables.
3. `ANALYZE` executed on key fact and dimension tables.
4. Materialized view created for daily revenue aggregation.

### Materialized View

`public.dw_mv_daily_revenue` pre-aggregates order revenue by date.

This improves performance for repeated dashboard queries such as:

- Daily revenue
- Number of orders per day
- Average order value by day

---

## 7. Sample Analytical Query Results

### Daily Revenue

{daily_revenue_md}

### Product Relationship Performance

{product_relationships_md}

---

## 8. Data Quality Checks

The following checks were executed to validate warehouse quality:

- Duplicate order IDs
- Duplicate event IDs
- Duplicate edge IDs
- Missing customer surrogate keys in order facts
- Missing customer/product surrogate keys in event facts

{dq_checks_md}

---

## 9. Analytics Capabilities

This warehouse supports the following analytical use cases:

- Daily, weekly, and monthly revenue reporting
- Average order value analysis
- Revenue by channel, campaign, and payment method
- Customer behavior and event funnel analysis
- Product performance and category analysis
- Recommendation graph relationship analysis
- Delivery performance and return analysis
- A/B variant behavior analysis
- Technical performance analysis using latency and dwell time metrics

---

## 10. Conclusion

The final Redshift warehouse centralizes heterogeneous data from PostgreSQL, Cassandra, and Neo4j into a clean dimensional model. The design provides a scalable foundation for business intelligence, customer analytics, product recommendations, and operational reporting.

The pipeline includes extraction, transformation, staging, dimensional loading, fact loading, quality validation, OLAP optimization, and final reporting outputs.
"""


# ---------------------------------------------------------
# 7. Save report
# ---------------------------------------------------------
report_path = os.path.join(BASE_DIR, "warehouse_report.md")

with open(report_path, "w") as f:
    f.write(report_content)

print(f"\n✅ Report saved to: {report_path}")

print("\n" + "="*60)
print("ALL TASKS COMPLETED! ✅")
print("="*60)


📄 Generating Final Report...


,table_name,row_count
0,dw_dim_ab_variant,2
1,dw_dim_browser,5
2,dw_dim_campaign,6
3,dw_dim_channel,5
4,dw_dim_customer,7057
5,dw_dim_date,552
6,dw_dim_device,3
7,dw_dim_os,5
8,dw_dim_payment_method,5
9,dw_dim_product,3360


,date_actual,revenue_usd,orders,avg_order_value
0,2024-01-01,4086.97,6,681.16
1,2024-01-02,2150.01,5,430.00
2,2024-01-03,681.74,6,113.62
3,2024-01-04,731.67,3,243.89
4,2024-01-05,1302.17,4,325.54
5,2024-01-06,4421.65,9,491.29
6,2024-01-07,2293.13,5,458.62
7,2024-01-09,1712.60,6,285.43
8,2024-01-10,2472.00,8,309.00
9,2024-01-11,4127.43,6,687.90


,relationship,category,edge_count,total_quantity,avg_edge_strength,avg_unit_price_usd
0,VIEWED,Sports,125,176,0.573,158.86
1,VIEWED,Beauty,93,125,0.547,141.11
2,VIEWED,Grocery,91,124,0.531,126.07
3,VIEWED,Apparel,88,117,0.508,135.68
4,VIEWED,Toys,86,133,0.589,158.55
5,VIEWED,Automotive,74,114,0.592,114.76
6,VIEWED,Home,70,98,0.537,91.11
7,VIEWED,Electronics,69,103,0.554,147.34
8,VIEWED,Books,69,92,0.579,147.77
9,PURCHASED,Sports,69,92,0.494,133.99


,check_name,issue_count
0,fact_orders_null_customer_sk,0
1,fact_events_null_product_sk,1118
2,fact_events_null_customer_sk,0
3,duplicate_event_id,0
4,duplicate_edge_id,0
5,duplicate_order_id,0



✅ Report saved to: ./warehouse_report.md

ALL TASKS COMPLETED! ✅
